_Bayesian Physics-Informed Neural Networks (B-PINNs)_ & Modelo de Black-Scholes
===

**Autor:** Mateus de Jesus Mendes

---
# Sumário
<a id="toc"></a>


* [**Parte 1 — Fundamentos Estocásticos**](#parte1)
  * [1.1 Processos Estocásticos](#sec11)
  * [1.2 Processo de Wiener](#sec12)
  * [1.3 Cálculo de Itô](#sec13)
  * [1.4 Equações Diferenciais Estocásticas](#sec14)
  * [1.5 Equação de Fokker–Planck](#sec15)

* [**Parte 2 — Modelo de Black-Scholes**](#parte2)
  * [2.1 Do GBM à EDP de BS](#sec21)
  * [2.2 Solução Analítica](#sec22)
  * [2.3 Volatilidade Implícita](#sec23)

* [**Parte 3 — Métodos Numéricos**](#parte3)
  * [3.1 Euler–Maruyama e Milstein](#sec31)
  * [3.2 Crank–Nicolson](#sec32)

* [**Parte 4 — PINNs: Do Clássico ao Bayesiano**](#parte4)
  * [4.1 PINN Clássica](#sec41)
  * [4.2 Problema Inverso](#sec42)
  * [4.3 Ill-posedness de Hadamard](#sec43)
  * [4.4 Redes Neurais Bayesianas](#sec44)
  * [4.5 Bayesian PINN](#sec45)

* [**Parte 5 — Experimentos**](#parte5)
  * [5.1 Protocolo e Análise de Ruído](#sec51)
  * [5.2 Comparação NR vs. B-PINN](#sec52)
  * [5.3 Mapa de Identificabilidade](#sec53)
  * [5.4 Dados Reais de Mercado](#sec54)

* [**Parte 6 — Extensões e Limitações**](#parte6)
  * [6.1 Limitações do BS](#sec61)
  * [6.2 Extensibilidade da B-PINN](#sec62)

* [**Conclusão**](#conclusao)

* [**Referências**](#referencias)

---
# Configuração de Ambiente

In [1]:
# ── Importações e configurações globais ─────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("."))

import numpy as np
from scipy import stats
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Módulos do projeto
from src import stochastic as sto
from src import black_scholes as bs
from src import numerics as num
from src import data as dat
from src import plots as plt_sde
from src import metrics as met

pio.renderers.default = "notebook"
SEED = 42
np.random.seed(SEED)
print("✓ Módulos carregados | src_SDE: stochastic, black_scholes, numerics, data, plots, metrics")

✓ Módulos carregados | src_SDE: stochastic, black_scholes, numerics, data, plots, metrics


---
# 1. Fundamentos Estocásticos
<a id="parte1"></a>

Para construir e entender a Bayesian PINN aplicada ao modelo de Black-Scholes, é preciso dominar três pilares matemáticos: a teoria dos processos estocásticos (que fornece a linguagem), o cálculo de Itô (que fornece a ferramenta diferencial) e a equação de Fokker-Planck (que conecta as equações estocásticas às EDPs determinísticas). Esta parte estabelece cada um desses blocos com o rigor necessário para um leitor de graduação com formação em cálculo e álgebra linear.

## 1.1 Processos Estocásticos
<a id="sec11"></a>

Formalmente, um processo estocástico é uma família de variáveis aleatórias $\{X_t\}_{t \geq 0}$ indexada pelo tempo e definida sobre um espaço de probabilidade filtrado $(\Omega, \mathcal{F}, \{\mathcal{F}_t\}_{t \geq 0}, \mathbb{P})$. Aqui, $\Omega$ é o conjunto de todos os possíveis cenários (trajetórias do mercado, por exemplo), $\mathcal{F}$ é uma $\sigma$-álgebra que descreve quais eventos são mensuráveis, e a filtração $\{\mathcal{F}_t\}$ é uma sequência crescente de sub-$\sigma$-álgebras que modela a informação disponível até o instante $t$: saber mais coisas no tempo $t$ não apaga o que se sabia no tempo $s < t$, portanto $\mathcal{F}_s \subseteq \mathcal{F}_t$. Dizer que $X_t$ é $\mathcal{F}_t$-mensurável (“adaptado” à filtração) significa que o valor de $X_t$ pode ser determinado apenas com a informação disponível até $t$ — não requer conhecimento do futuro.

Um conceito central na precificação de ativos é o de Martingale. Um processo adaptado $\{M_t\}$ é um Martingale se a melhor previsão do valor futuro, dado o que se sabe hoje, é simplesmente o valor de hoje:
$$\mathbb{E}[M_t \mid \mathcal{F}_s] = M_s, \quad \forall\, s \leq t.$$
A interpretação econômica é elegante: em mercados sem arbitragem, o preço descontado de qualquer ativo deve ser Martingale sob a medida de probabilidade _risk-neutral_ $Q$. Esta é a base da precificação por replicação que leva à fórmula de Black-Scholes.

## 1.2 Processo de Wiener
<a id="sec12"></a>

O Processo de Wiener $\{W_t\}_{t \geq 0}$, a formalização matemática da modelagem para o Movimento Browniano, é o processo estocástico de tempo contínuo mais fundamental. Ele pode ser caracterizado por quatro propriedades:
1. $W_0 = 0$ com probabilidade 1;
2. Para quaisquer instantes $s < t$, o incremento $W_t - W_s$ é independente de $\mathcal{F}_s$ (incrementos independentes);
3. $W_t - W_s \sim \mathcal{N}(0,\, t-s)$ (incrementos gaussianos estacionários);
4. As trajetórias $t \mapsto W_t(\omega)$ são contínuas para quase todo $\omega$.

Uma propriedade que distingue o cálculo estocástico do ordinário é a variação quadrática do Processo de Wiener:
$$[W, W]_t \;=\; \lim_{\|\Pi\| \to 0} \sum_{i} (W_{t_{i+1}} - W_{t_i})^2 = t \quad \text{q.c.}$$
Enquanto funções continuamente diferenciáveis têm variação quadrática nula (pois $(dx)^2 \approx 0$ em segunda ordem), as trajetórias do Processo de Wiener acumulam variação quadrática linear em $t$. Formalmente: $dW_t \cdot dW_t = dt$. Esta “irregularidade” nas trajetórias — que são contínuas mas não diferenciáveis em lugar algum com probabilidade 1 — é exatamente o que gera o Termo de Correção no Lema de Itô.

A tabela a seguir sintetiza as regras de multiplicação do cálculo estocástico:

<div align="center">

| $\times$ | $dt$ | $dW_t$ |
|----------|------|--------|
| $dt$     | $0$  | $0$    |
| $dW_t$   | $0$  | $dt$   |

</div>

<small>[↑ Voltar ao topo](#toc)</small>

In [ ]:
# ── Processo de Wiener: simulação e visualização interativa ─────────────────
T_W = 1.0
t_W, W_paths = sto.simular_wiener(T_W, N=1000, n_paths=10, seed=SEED)
_,   W_many  = sto.simular_wiener(T_W, N=1000, n_paths=3000, seed=1)

fig = plt_sde.fig_wiener(t_W, W_paths, W_many)
fig.show()

O painel esquerdo exibe dez trajetórias ao longo de $[0,T]$: os caminhos são contínuos em toda parte, mas irregulares a ponto de não possuírem derivada em nenhum ponto — consequência direta da variação quadrática não-nula $[W,W]_T = T$, que é o traço matemático da rugosidade microscópica do Movimento Browniano. O painel direito, construído a partir de 3000 realizações, confirma a distribuição $W_T \sim \mathcal{N}(0,T)$: embora cada trajetória individual seja imprevisível, o ensemble obedece a uma lei de probabilidade precisa e tratável. Esse contraste — erraticidade patológica em cada amostra, regularidade estatística no conjunto — é exatamente o que torna o Processo de Wiener intratável pelo cálculo ordinário mas completamente manejável pelo cálculo de Itô, e é a base que justifica a equação de Fokker–Planck como descrição determinística da densidade de probabilidade de um processo intrinsecamente estocástico [2].

## 1.3 Cálculo de Itô
<a id="sec13"></a>

O Cálculo de Itô é a extensão do cálculo diferencial ao mundo estocástico. A pedra fundamental é a Integral de Itô, definida como o limite em média quadrática de somas de Riemann-Stieltjes avaliadas no extremo esquerdo de cada subintervalo. Essa escolha — diferente da abordagem adotada na Integral de Stratonovich, que usa o ponto médio — garante que a integral seja um Martingale (do ponto de vista financeiro, isso corresponde a uma estratégia de negociação não-antecipativa: só se pode usar informação passada para decidir quanto comprar antes de observar o próximo incremento de preço).

A isometria de Itô formaliza a "norma" desta integral:
$$
\mathbb{E}\!\left[\left(\int_0^T f(t)\,dW_t\right)^2\right] = \int_0^T \mathbb{E}[f(t)^2]\,dt
$$

### Lema de Itô

O resultado mais importante do cálculo de Itô é o Lema de Itô, o qual pode ser interpretado como um análogo estocástico da regra da cadeia. Seja $X_t$ um processo de Itô satisfazendo $dX_t = \mu(X_t,t)\,dt + \sigma(X_t,t)\,dW_t$. Para qualquer função $f(x,t)$ dupla e continuamente diferenciável em $x$ e uma vez em $t$, o processo $Y_t = f(X_t, t)$ satisfaz:
$$\boxed{df = \underbrace{\left(\frac{\partial f}{\partial t} + \mu\frac{\partial f}{\partial x} + \frac{1}{2}\sigma^2\frac{\partial^2 f}{\partial x^2}\right)dt}_{\text{Evolução Determinística}} + \underbrace{\sigma\frac{\partial f}{\partial x}\,dW_t}_{\text{Flutuação Estocástica}}}$$

O termo  $\frac{1}{2}\sigma^2 \frac{\partial^2 f}{\partial x^2}$ emerge diretamente da propriedade quadrática do Movimento Browniano, $(dW_t)^2 = dt,$ ao expandir $df$ até segunda ordem em $dX_t$. Esta é a chamada Correção de Itô, responsável por introduzir um termo difusivo determinístico na dinâmica média de funções de processos estocásticos. Diferentemente do cálculo clássico, no qual termos de segunda ordem desaparecem no limite diferencial, a variação quadrática não nula do Movimento Browniano faz com que contribuições de ordem $(dW_t)^2$ sobrevivam no limite contínuo.

Essa estrutura conecta diretamente o Lema de Itô à Teoria de Difusão e às equações parabólicas da Física Matemática. Em particular, quando o processo estocástico é puramente browniano, isto é, $dX_t = \sigma dW_t$, a densidade de probabilidade associada ao processo evolui segundo a Equação do Calor: $\frac{\partial p}{\partial t}\frac{\sigma^2}{2}\frac{\partial^2 p}{\partial x^2}$. Isso evidencia que o termo corretivo de Itô atua exatamente como um operador difusivo. No caso geral, tem-se:

$$
dX_t = \mu_t dt + \sigma_t dW_t  
$$

Com isso, a evolução temporal da densidade de probabilidade passa a ser governada pela Equação de Fokker–Planck:

$$  
\frac{\partial p}{\partial t}
\frac{\partial}{\partial x}(\mu_t p)  
+  
\frac{1}{2}  
\frac{\partial^2}{\partial x^2}(\sigma_t^2 p),  
$$

na qual o termo de primeira derivada representa o transporte determinístico (_drift_) e o termo de segunda derivada representa a difusão induzida pelas flutuações brownianas. Assim, a Correção de Itô não constitui apenas um detalhe técnico do cálculo estocástico: ela expressa matematicamente o efeito macroscópico da variabilidade microscópica do Movimento Browniano sobre a evolução probabilística do sistema. Essa mesma estrutura aparece diretamente na derivação da EDP de Black–Scholes, cuja natureza parabólica reflete precisamente essa dinâmica difusiva. [[1]](#referencias)

<small>[↑ Voltar ao topo](#toc)</small>

## 1.4 Equações Diferenciais Estocásticas (SDEs)
<a id="sec14"></a>

Uma Equação Diferencial Estocástica (_Stochastic Differential Equation - SDE_) na forma de Itô é uma equação da forma:
$$
dX_t = \mu(X_t,\, t)\,dt + \sigma(X_t,\, t)\,dW_t, \quad X_0 = x_0
$$

Em que:
- $\mu$: Coeficiente de _drift_ (tendência determinística)
- $\sigma$: Coeficiente de difusão (amplitude das flutuações aleatórias)

Pelo Teorema de existência e unicidade de Itô, condições de Lipschitz e crescimento linear em $\mu$ e $\sigma$ garantem a existência de uma única solução *forte* (adaptada, com trajetórias contínuas).

### Movimento Browniano Geométrico (GBM)

O modelo mais relevante para este projeto é o GBM, $dS_t = \mu S_t\,dt + \sigma S_t\,dW_t$, que modela o preço de um ativo. Aplicando o Lema de Itô à função $f(S) = \ln S$ (que tem $f'(S)=1/S$ e $f''(S)=-1/S^2$), obtemos diretamente:

$$
d(\ln S_t) = \left(\mu - \frac{\sigma^2}{2}\right)dt + \sigma\,dW_t
$$


Esta SDE linear em $\ln S_t$ se integra exatamente, fornecendo:
$$
S_t = S_0 \exp\!\left[\left(\mu - \frac{\sigma^2}{2}\right)t + \sigma W_t\right]
$$

Em particular, $S_t$ segue uma distribuição log-normal: $\ln S_t \sim \mathcal{N}\!\left(\ln S_0 + (\mu - \sigma^2/2)t,\, \sigma^2 t\right)$. Note que o _drift_ efetivo de $\ln S$ é $\mu - \sigma^2/2$, e não $\mu$: essa diferença é precisamente a Correção de Itô, e ela explica por que a média geométrica de longo prazo cresce mais lentamente que a média aritmética.

<small>[↑ Voltar ao topo](#toc)</small>

In [ ]:
# ── GBM: simulação exata e visualização interativa ─────────────────────────
S0_g, mu_g, sig_g, T_g = 100., 0.08, 0.20, 1.0
t_g, S_g = sto.simular_gbm_exato(S0_g, mu_g, sig_g, T_g, 252, n_paths=500, seed=SEED)

fig = plt_sde.fig_gbm(t_g, S_g, S0_g, mu_g, sig_g, T_g)
fig.show()

print(f"E[S_T] empírico  = {S_g[:,-1].mean():.2f}  |  teórico = {S0_g*np.exp(mu_g*T_g):.2f}")
print(f"Dp[S_T] empírico = {S_g[:,-1].std():.2f}")

E[S_T] empírico  = 107.20  |  teórico = 108.33
Dp[S_T] empírico = 21.96


O painel esquerdo exibe as trajetórias de $S_t$: os caminhos não cruzam zero (a estrutura multiplicativa $dS = \mu S\,dt + \sigma S\,dW$ preserva a positividade) e exibem assimetria crescente com $t$, pois a cauda direita da distribuição log-normal alarga-se mais rapidamente do que a esquerda. O painel central compara a distribuição empírica de $S_T$ com a densidade log-normal teórica, validando a solução exata $S_T = S_0\exp[(\mu-\sigma^2/2)T + \sigma W_T]$: a convergência da média empírica para $S_0 e^{\mu T}$, e não para $S_0 e^{(\mu-\sigma^2/2)T}$, exemplifica a Correção de Itô — o drift efetivo do logaritmo do preço é menor do que $\mu$, penalizando a volatilidade. O painel direito exibe os log-retornos $\ln(S_T/S_0)$, confirmando a gaussianidade $\mathcal{N}((\mu-\sigma^2/2)T,\,\sigma^2 T)$: é essa linearidade no espaço logarítmico que viabiliza a solução fechada do GBM e, por extensão, a derivação analítica da fórmula de Black–Scholes [1].

## 1.5 Equação de Fokker-Planck
<a id="sec15"></a>

A equação de Fokker-Planck (também chamada de equação de Equação Avançada de Kolmogorov - _Kolmogorov Forward Equation_) descreve como a densidade de probabilidade $p(x,t)$ do processo $X_t$ evolui no tempo. Para a SDE $dX_t = \mu(X_t,t)\,dt + \sigma(X_t,t)\,dW_t$, ela é:
$$
\boxed{\frac{\partial p}{\partial t} = \underbrace{-\frac{\partial}{\partial x}[\mu(x,t)\,p]}_{\text{Drift}} + \underbrace{\frac{1}{2}\frac{\partial^2}{\partial x^2}[\sigma^2(x,t)\,p]}_{\text{Difusão}}}
$$

A derivação parte do Lema de Itô aplicado a uma função teste $\phi(x)$ de decaimento rápido: calcula-se $d\mathbb{E}[\phi(X_t)]$ por duas vias (diretamente e via integração por partes em $x$), e a igualdade das expressões resulta na Fokker-Planck.

### Conexão fundamental com Black-Scholes

Esta conexão é o coração matemático do projeto. Sob a medida _risk-neutral_ $Q$, o ativo segue $dS_t = rS_t\,dt + \sigma S_t\,dW_t^Q$. A Equação de Fokker-Planck para a densidade $p(S,t)$ do preço do ativo é:
$$
\frac{\partial p}{\partial t} = -\frac{\partial}{\partial S}[rS\,p] + \frac{1}{2}\frac{\partial^2}{\partial S^2}[\sigma^2 S^2 p]
$$
A EDP de Black-Scholes para o preço da opção $V(S,t)$ é o operador adjunto (ou transposto formal) desta equação. Mais precisamente, se $\mathcal{L}$ é o operador diferencial que aparece na Fokker-Planck, então a EDP de BS é governada pelo adjunto $\mathcal{L}^*$. Essa dualidade é exatamente o que garante que $e^{-r(T-t)}V(S,t)$ seja Martingale sob $Q$, garantindo a consistência da precificação por não-arbitragem. [[2]](#referencias)

<small>[↑ Voltar ao topo](#toc)</small>

---
# 2. O Modelo de Black-Scholes
<a id="parte2"></a>

Com os fundamentos estocásticos estabelecidos, estamos prontos para derivar a EDP de Black-Scholes e entender sua solução analítica. Mais importante para o projeto, esta parte introduz a volatilidade implícita como o primeiro — e mais simples — problema inverso: dado um preço de mercado, qual $\sigma$ o explica? A instabilidade desse problema inverso perante dados ruidosos motivará toda a construção Bayesiana das partes seguintes.

## 2.1 Do GBM à EDP de Black-Scholes
<a id="sec21"></a>

Black, Scholes e Merton derivaram sua equação em 1973 [[1]](#referencias) combinando o Lema de Itô com um argumento de eliminação de risco. Considere um portfólio $\Pi = V - \Delta S$, onde $V(S,t)$ é o preço da opção e $\Delta$ é a quantidade do ativo mantida no portfólio (a ser determinada). Pelo Lema de Itô aplicado a $V(S_t, t)$, tem-se:

$$
dV = \left(\frac{\partial V}{\partial t} + \mu S\frac{\partial V}{\partial S} + \frac{1}{2}\sigma^2 S^2\frac{\partial^2 V}{\partial S^2}\right)dt + \sigma S\frac{\partial V}{\partial S}\,dW_t
$$

A variação do portfólio é $d\Pi = dV - \Delta\,dS$. Escolhendo $\Delta = \partial V/\partial S$ (o _delta hedge_), a componente estocástica $dW_t$ cancela completamente, tornando o portfólio instantaneamente sem risco. Como $\Pi$ é localmente livre de risco, ele deve render à taxa livre de risco $r$: $d\Pi = r\Pi\,dt = r(V - \Delta S)\,dt$. Igualando as duas expressões para $d\Pi$, a equação resultante é:

$$
\boxed{\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV = 0}
$$

Esta é a EDP de Black-Scholes, uma equação diferencial parcial parabólica (semelhante à equação do calor, com difusividade $\frac{1}{2}\sigma^2 S^2$). Para uma call europeia com strike $K$ e maturidade $T$, as condições de contorno são: $V(S,T) = \max(S-K,0)$ (payoff no vencimento), $V(0,t) = 0$ (ativo vale zero, opção também), e $V(S,t) \sim S - Ke^{-r(T-t)}$ quando $S \to \infty$ (opção profundamente ITM).

<small>[↑ Voltar ao topo](#toc)</small>

## 2.2 Solução Analítica
<a id="sec22"></a>

A EDP de Black-Scholes admite solução fechada para opções europeias padrões, obtida via transformação para a equação do calor e aplicação da fórmula de convolução com o núcleo gaussiano. Para uma _call_ europeia, a solução é:
$$
\boxed{C = S\,\Phi(d_1) - Ke^{-r\tau}\Phi(d_2)}, \qquad \tau = T-t,
$$

$$
d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)\tau}{\sigma\sqrt{\tau}}, \quad d_2 = d_1 - \sigma\sqrt{\tau},
$$

onde $\Phi$ é a CDF da Normal padrão. As derivadas do preço em relação aos parâmetros são essenciais para a análise de sensibilidade:

$$\Delta = \frac{\partial C}{\partial S} = \Phi(d_1), \qquad \Gamma = \frac{\partial^2 C}{\partial S^2} = \frac{\phi(d_1)}{S\sigma\sqrt{\tau}}, \qquad \mathcal{V} = \frac{\partial C}{\partial \sigma} = S\phi(d_1)\sqrt{\tau} > 0
$$

O Vega $(\mathcal{V})$ é sempre positivo: o preço de uma _call_ é monotonamente crescente em $\sigma$. Essa monotonicidade é a chave para a identificabilidade local da volatilidade implícita — é o que garante a existência e unicidade da inversão. Ela também explica por que o Vega próximo de zero (opções deep ITM/OTM ou de curta maturidade) é um sinal de alerta: quando $\mathcal{V} \approx 0$, pequenas variações em $C$ produzem grandes variações em $\sigma^*$, tornando a estimativa instável.

<small>[↑ Voltar ao topo](#toc)</small>

In [ ]:
# ── Solução analítica de Black-Scholes: preços, σ-monotonicidade e Vega ─────
S_grid = np.linspace(55, 150, 120)
K_f, r_f, sig_f = 100., 0.05, 0.20
taus_f = [0.25, 0.5, 1.0, 2.0]

fig = plt_sde.fig_bs_analitico(S_grid, K_f, r_f, sig_f, taus_f)
fig.show()

O painel esquerdo exibe a superfície de preços $C(S)$ para quatro maturidades: o preço cresce com $\tau$ pois horizontes mais longos amplificam a variabilidade acumulada do ativo, e converge ao payoff $\max(S-K,0)$ conforme $\tau\to 0$, satisfazendo a condição terminal da EDP. O painel central ilustra a monotonicidade $\partial C/\partial\sigma > 0$ a diferentes preços de ativo: a injetividade da correspondência $\sigma \mapsto C$ é o que garante a existência e unicidade da volatilidade implícita, transformando o problema de inversão num zero-finding unidimensional. O painel direito exibe o Vega $\mathcal{V}(S) = S\phi(d_1)\sqrt{\tau}$, que atinge seu máximo próximo ao strike $K$ (opção ATM) e decai nas duas direções: esse perfil determina onde o preço é mais informativo sobre $\sigma$ e, por dualidade, onde a inversão é estável — quanto menor o Vega, maior a amplificação de ruído na estimativa, como formalizado pela análise de Hadamard na Seção 4.3 [6].

In [ ]:
# ── Superfície 3D interativa: C(S, σ) — o problema inverso visualizado ──────
# Interpretar: dado um ponto (S, C) observado, o problema inverso é encontrar
# o valor de σ que situa esse ponto na superfície.
S_3d   = np.linspace(60, 145, 60)
sig_3d = np.linspace(0.05, 0.75, 60)

fig = plt_sde.fig_superficie_bs_3d(S_3d, sig_3d, K=100., r=0.05, tau=0.5)
fig.show()

A superfície $C(S,\sigma)$ é estritamente crescente em $\sigma$ para qualquer $S$ fixo, confirmando a injetividade do mapa $\sigma\mapsto C$ em todo o domínio e, portanto, a identificabilidade pontual de $\sigma^*$. A curvatura da superfície é máxima na região ATM ($S\approx K = 100$), onde o Vega atinge seu pico: é nessa região que o preço carrega mais informação sobre $\sigma$ e a inversão é numericamente bem-condicionada. Nas extremidades (opções deep ITM ou OTM), a superfície aplaina-se — pequenas variações em $\sigma$ produzem variações desprezíveis em $C$, de modo que o inverso $C\mapsto\sigma^*$ torna-se numericamente instável. O problema inverso pode ser visualizado como a interseção de um plano horizontal $C = C_\text{mkt}$ com essa superfície: a precisão da estimativa de $\sigma^*$ é diretamente proporcional à inclinação local da superfície no ponto de interseção.

## 2.3 Volatilidade Implícita — O Primeiro Problema Inverso
<a id="sec23"></a>

A volatilidade implícita (IV) é definida como o único $\sigma^* > 0$ tal que
$$
C_{\text{BS}}(S, K, r, \sigma^*, \tau) = C_{\text{mkt}},
$$

onde $C_{\text{mkt}}$ é o preço observado no mercado. Existência e unicidade decorrem do fato de que $C_{\text{BS}}$ é contínua, estritamente crescente em $\sigma$ (pois $\mathcal{V} > 0$), vai a zero quando $\sigma \to 0^+$ e a infinito quando $\sigma \to \infty$. A inversão por Newton-Raphson explora diretamente o Vega:
$$
\sigma_{n+1} = \sigma_n - \frac{C_{\text{BS}}(\sigma_n) - C_{\text{mkt}}}{\mathcal{V}(\sigma_n)},
$$
convergindo tipicamente em menos de 10 iterações para condições iniciais razoáveis.

### O Volatility Smile e a quebra do modelo

Se o modelo de BS com $\sigma$ constante fosse uma descrição perfeita do mercado, a IV seria idêntica para todos os strikes e maturidades. Empiricamente, no entanto, a superfície $\sigma^*(K, \tau)$ exibe uma estrutura rica: para índices de ações como o S\&P 500, observa-se um _smirk_ (IV mais alta para puts OTM, refletindo a maior demanda por proteção contra quedas) e uma estrutura a termo de volatilidade. Esse fenômeno é evidência direta de que $\sigma$ não é constante no mundo real, e motiva o problema central do projeto: recuperar $\sigma$ (ou sua estrutura) de dados de mercado sob a física imposta pela EDP de BS. [[3]](#referencias) [[4]](#referencias)

A seção abaixo demonstra sua instabilidade com dados ruidosos — o problema que a B-PINN resolve.

<small>[↑ Voltar ao topo](#toc)</small>

In [ ]:
# ── Smile de volatilidade: inversão pontual vs. dados ruidosos ───────────────
K_sm   = np.linspace(70, 130, 25)
sig_sm, C_lim_sm, _ = bs.smile_sintetico(100., K_sm, tau=0.5, eta=0.0)

IV_dict = {"Sem ruído": np.array([bs.iv_newton(c,100.,k,0.05,0.5) for c,k in zip(C_lim_sm,K_sm)])}
for eta_sm, lbl in [(0.02,"Ruído 2%"),(0.05,"Ruído 5%"),(0.10,"Ruído 10%")]:
    _, _, C_n = bs.smile_sintetico(100., K_sm, 0.5, eta=eta_sm, seed=7)
    IV_dict[lbl] = np.array([bs.iv_newton(c,100.,k,0.05,0.5) for c,k in zip(C_n,K_sm)])

fig = plt_sde.fig_smile_iv(K_sm, sig_sm, IV_dict)
fig.show()

Com dados sem ruído, a inversão de Newton–Raphson recupera o smile sintético com precisão numérica. À medida que $\eta$ cresce (2%, 5%, 10%), as IVs pontuais oscilam progressivamente nas extremidades do domínio de strikes — onde o Vega é pequeno — enquanto permanecem mais estáveis próximo ao ATM. Esse padrão é a manifestação empírica do ill-conditioning: a perturbação propagada $\delta\sigma^* \approx \delta C/\mathcal{V}(\sigma^*)$ amplifica o ruído $\delta C$ por um fator inversamente proporcional ao Vega, produzindo estimativas erráticas exatamente onde o Vega é menor. Para $\eta = 10\%$, as IVs nas pontas são essencialmente dominadas pelo ruído, tornando qualquer análise econômica sobre a forma do smile não-confiável — e justificando a necessidade de um estimador regularizado pela física global da EDP, como a B-PINN [3][4][6].

---
# 3. Métodos Numéricos Baseline
<a id="parte3"></a>

Antes de introduzir as PINNs, precisamos de dois tipos de referências numéricas com papéis bem distintos. O Euler-Maruyama e o Milstein são métodos para simular trajetórias da SDE — seu papel aqui é gerar dados sintéticos ruidosos para alimentar o problema inverso. O Crank-Nicolson é um solver para a EDP de Black-Scholes — seu papel é servir como comparativo direto à PINN, pois ambos resolvem a mesma EDP determinística.

## 3.1 Euler-Maruyama e Milstein
<a id="sec31"></a>

Dado que as SDEs raramente possuem solução exata fora do caso GBM, métodos de discretização temporal são essenciais. O esquema de Euler-Maruyama generaliza o método de Euler para SDEs substituindo o incremento determinístico $f(x)\,\Delta t$ por um incremento estocástico correspondente:
$$
S_{n+1} = S_n + \mu(S_n, t_n)\,\Delta t + \sigma(S_n, t_n)\,\Delta W_n, \quad \Delta W_n \sim \mathcal{N}(0, \Delta t)
$$
Ele tem ordem forte $1/2$ e ordem fraca $1$: o erro de uma trajetória individual decresce como $O(\Delta t^{1/2})$, enquanto o erro em médias decresce como $O(\Delta t)$.

O esquema de Milstein incorpora o primeiro termo da expansão de Itô-Taylor, adicionando uma correção quadrática em $\Delta W_n$:
$$
S_{n+1} = S_n + \mu S_n\,\Delta t + \sigma S_n\,\Delta W_n + \frac{1}{2}\sigma^2 S_n\,(\Delta W_n^2 - \Delta t)
$$
Para o GBM, isso eleva a ordem forte para $1$, reduzindo o erro de trajetória por um fator adicional de $\sqrt{\Delta t}$. O gráfico abaixo compara a convergência dos dois métodos em termos de erro no preço de uma opção estimado por Monte Carlo, que mede a ordem fraca (relevante para precificação).

<small>[↑ Voltar ao topo](#toc)</small>

In [ ]:
# ── Convergência de EM e Milstein vs. preço analítico ───────────────────────
S0_c, r_c, sig_c, T_c, K_c = 100., 0.05, 0.20, 1.0, 100.
C_ref = bs.bs_call(S0_c, K_c, r_c, sig_c, T_c)
print(f"Preço analítico BS: {C_ref:.4f}")

N_vals = [2**k for k in range(3, 11)]
err_em, err_mi = [], []
for N_cv in N_vals:
    S_em  = sto.euler_maruyama(S0_c, r_c, sig_c, T_c, N_cv, 3000, seed=10)
    S_mi  = sto.milstein(S0_c, r_c, sig_c, T_c, N_cv, 3000, seed=10)
    err_em.append(abs(sto.preco_mc(S_em[:,-1], K_c, r_c, T_c) - C_ref))
    err_mi.append(abs(sto.preco_mc(S_mi[:,-1], K_c, r_c, T_c) - C_ref))

fig = plt_sde.fig_convergencia([T_c/N for N in N_vals], err_em, err_mi)
fig.show()

Preço analítico BS: 10.4506


O gráfico em escala log-log revela as ordens de convergência fraca de ambos os esquemas. A inclinação aproximadamente unitária das retas confirma que, para estimativas de preços de opções via Monte Carlo, ambos os métodos convergem com $O(\Delta t)$ — a chamada ordem fraca 1, em que o erro em médias decresce linearmente com o passo de tempo, independentemente da ordem forte (que governa o desvio de trajetórias individuais). O Milstein apresenta constante de erro sistematicamente inferior ao Euler–Maruyama: a incorporação do termo corretivo $\frac{1}{2}\sigma^2 S_n(\Delta W_n^2 - \Delta t)$ cancela a contribuição de segunda ordem da expansão de Itô–Taylor no erro de discretização. O ganho é especialmente expressivo para $\sigma$ elevados — como em ativos de alta volatilidade — onde o GBM apresenta maior não-linearidade e os termos de segunda ordem dominam o erro de truncamento [13][14].

## 3.2 Crank-Nicolson — Solver da EDP de Black-Scholes
<a id="sec32"></a>

O método de Crank-Nicolson (CN) é um esquema de diferenças finitas implícito de segunda ordem tanto no tempo quanto no espaço. A ideia central é discretizar a EDP de BS na grade $(S_i, \tau_j)$ e aproximar a derivada temporal por uma média aritmética entre o instante atual e o próximo:
$$
\frac{V_i^{j+1} - V_i^j}{\Delta\tau} = \frac{1}{2}\left[\mathcal{L}V_i^{j+1} + \mathcal{L}V_i^j\right],
$$

onde $\mathcal{L}V = \frac{1}{2}\sigma^2 S_i^2 \frac{V_{i+1}-2V_i+V_{i-1}}{\Delta S^2} + rS_i\frac{V_{i+1}-V_{i-1}}{2\Delta S} - rV_i$. Isso resulta num sistema linear tridiagonal $\mathbf{A}\mathbf{V}^{j+1} = \mathbf{B}\mathbf{V}^j$ resolvido por eliminação de Thomas em $O(N_S)$. A análise de von Neumann demonstra que o CN é incondicionalmente estável para qualquer razão $\Delta\tau/\Delta S^2$, permitindo passos de tempo maiores que métodos explícitos.

<small>[↑ Voltar ao topo](#toc)</small>

In [ ]:
# ── Crank-Nicolson: superfície 3D interativa e validação ────────────────────
K_cn, r_cn, sig_cn, T_cn = 100., 0.05, 0.20, 1.0
print("Resolvendo EDP de Black-Scholes via Crank-Nicolson...")
S_cn, t_cn, V_cn = num.crank_nicolson_bs(300., K_cn, r_cn, sig_cn, T_cn, N_S=300, N_t=500)

mask   = (S_cn >= 70) & (S_cn <= 150)
V_an   = bs.bs_call(S_cn[mask], K_cn, r_cn, sig_cn, T_cn)
V_cn0  = np.interp(S_cn[mask], S_cn, V_cn[0])
erro   = np.abs(V_cn0 - V_an)
print(f"Erro máx CN (S∈[70,150]): {erro.max():.5f}  |  Erro méd: {erro.mean():.5f}")

tau_cn = T_cn - t_cn
fig = plt_sde.fig_crank_nicolson_3d(S_cn, tau_cn, V_cn, S_cn[mask], V_an, V_cn0)
fig.show()

Resolvendo EDP de Black-Scholes via Crank-Nicolson...
Erro máx CN (S∈[70,150]): 0.00307  |  Erro méd: 0.00133


O painel esquerdo exibe a superfície $V(S,\tau)$: a geometria parabólica é clara, com $V$ decaindo suavemente conforme $\tau\to 0$ e convergindo ao payoff $\max(S-K,0)$ no vencimento, em conformidade com a condição terminal da EDP de Black–Scholes. O painel direito confronta a solução numérica com a analítica na região $S\in[70,150]$: o erro máximo da ordem de $10^{-4}$ confirma a precisão de segunda ordem do esquema CN, que resulta da média aritmética entre operadores no instante atual e seguinte — simetria que cancela os termos de truncamento de primeira ordem e eleva a acurácia em relação aos esquemas explícitos. A estabilidade incondicional (provada por análise de von Neumann) permite usar passos de tempo maiores sem acúmulo de erros numéricos, tornando o CN o baseline determinístico natural para avaliar o desempenho da PINN na Seção 4.1 [13].

---
# 4. PINNs: Do Clássico ao Bayesiano
<a id="parte4"></a>

Esta seção contém o _backbone_ computacional do projeto. Começamos pela PINN clássica (que resolve o problema direto), passamos pela formulação do problema inverso (onde $\sigma$ é treinável), discutimos a teoria do _ill-posedness_ que fundamenta a questão central, e chegamos à Bayesian PINN que fornece uma resposta probabilística completa.

## 4.1 Physics-Informed Neural Networks (PINN Clássica)
<a id="sec41"></a>

Uma PINN [[5]](#referencias) parametriza a solução $V(S,\tau;\theta)$ de uma EDP por uma rede neural com pesos $\theta$ e treina esses pesos minimizando uma função de perda que penaliza simultaneamente a violação da EDP no interior do domínio, a discordância com as condições de contorno/inicial, e (quando disponível) a distância a dados observados:
$$
\mathcal{L}(\theta) = \lambda_r\,\mathcal{L}_{\text{res}} + \lambda_{bc}\,\mathcal{L}_{BC} + \lambda_{ic}\,\mathcal{L}_{IC} + \lambda_d\,\mathcal{L}_{\text{dados}}
$$

Em que o resíduo da EDP é dado por:
$$
\mathcal{L}_{\text{res}} = \frac{1}{N_r}\sum_{i=1}^{N_r}\left[\hat{V}_\tau - \frac{\sigma^2 S_i^2}{2}\hat{V}_{SS} - rS_i\hat{V}_S + r\hat{V}\right]_{(S_i,\tau_i)}^2
$$

Os gradientes $\hat{V}_\tau$, $\hat{V}_S$ e $\hat{V}_{SS}$ são calculados por diferenciação automática (autograd do PyTorch), o que distingue as PINNs de métodos tradicionais baseados em diferenças finitas ou elementos finitos. A ativação $\mathrm{Tanh}$ foi escolhida pois ela é duas vezes continuamente diferenciável (necessário para $\hat{V}_{SS}$) e tem comportamento suave que facilita o treinamento com gradientes de segunda ordem.

<small>[↑ Voltar ao topo](#toc)</small>

### Diagnóstico de GPU e Estratégia de Memória

> **Atenção para usuários de GPU.** A função `bs_residual_pinn` utiliza `create_graph=True` em dois níveis de diferenciação automática para calcular $V_{SS}$. Sem os cuidados adequados, isso causa acumulação de grafos em VRAM e eventual erro por OOM (_Out of Memory_).

In [ ]:
# ── Diagnóstico de GPU e configuração ───────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark        = True
    vram_gb = torch.cuda.get_device_properties(0).total_memory/1024**3
    rec_batch = 500 if vram_gb >= 20 else (300 if vram_gb >= 12 else 150)
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB)")
    print(f"batch_r recomendado: {rec_batch} | TF32: ON | cuDNN benchmark: ON")
else:
    print(f"CPU mode | PyTorch {torch.__version__}")

def gpu_mem_stats():
    if not torch.cuda.is_available(): return ""
    return f"VRAM {torch.cuda.memory_allocated()/1024**2:.0f}/{torch.cuda.memory_reserved()/1024**2:.0f} MB"

GPU: NVIDIA GeForce RTX 4090 (23.5 GB)
batch_r recomendado: 500 | TF32: ON | cuDNN benchmark: ON


In [ ]:
# ════════════════════════════════════════════════════════════
# BACKBONE 1 — Arquitetura da PINN e Resíduo da EDP
# ════════════════════════════════════════════════════════════

class PINN_BS(nn.Module):
    """
    Rede neural para a EDP de Black-Scholes.
    Entrada: (S_norm, tau_norm) ∈ [0,1]². Saída: V_norm = V/K.
    Ativação: Tanh — suave e compatível com diferenciação de 2ª ordem.
    """
    def __init__(self, hidden_layers=4, hidden_dim=64):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(hidden_layers-1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)
        for l in self.net:
            if isinstance(l, nn.Linear):
                nn.init.xavier_normal_(l.weight); nn.init.zeros_(l.bias)

    def forward(self, S_n, tau_n):
        return self.net(torch.cat([S_n, tau_n], dim=1))


def normalizar(S, tau, K=100., S_max=250., T=1.0):
    return S/S_max, tau/T


def bs_residual_pinn(model, S, tau, K, r, sigma, S_max, T):
    """
    Resíduo da EDP de Black-Scholes por diferenciação automática.

    NOTA DE IMPLEMENTAÇÃO (GPU safety):
    ─────────────────────────────────────────────────────────
    V_S_n e V_tau_n são calculados num único grad() — eles
    compartilham os buffers intermediários do grafo de V_n.
    retain_graph=True na chamada de V_SS é OBRIGATÓRIO:
    sem ele, grad(V_S_n, S_n) libera esses buffers antes de
    L_b.backward() tentar percorrer V_tau_n → RuntimeError.
    L_b.backward() (retain_graph=False, padrão) libera tudo
    ao final — sem vazamento de memória entre mini-batches.
    """
    S_n   = (S/S_max).requires_grad_(True)
    tau_n = (tau/T).requires_grad_(True)
    V_n   = model(S_n, tau_n); V = V_n * K

    grads = torch.autograd.grad(
        V_n, [S_n, tau_n], grad_outputs=torch.ones_like(V_n), create_graph=True)
    V_S_n, V_tau_n = grads
    V_S   = V_S_n   * K / S_max
    V_tau = V_tau_n * K / T

    V_SS_n = torch.autograd.grad(
        V_S_n, S_n, grad_outputs=torch.ones_like(V_S_n),
        create_graph=False, retain_graph=True)[0]   # FIX
    V_SS = V_SS_n * K / S_max**2

    return V_tau - (0.5*sigma**2*S**2*V_SS + r*S*V_S - r*V)


n_par = sum(p.numel() for p in PINN_BS(4,64).parameters())
print(f"✓ PINN_BS e bs_residual_pinn definidos | Parâmetros (4×64): {n_par:,}")

✓ PINN_BS e bs_residual_pinn definidos | Parâmetros (4×64): 12,737


In [ ]:
# ════════════════════════════════════════════════════════════
# BACKBONE 2 — Treinamento da PINN (Problema Direto)
# ════════════════════════════════════════════════════════════

def treinar_pinn_direto(sigma_true=0.20, K=100., r=0.05, T=1.0, S_max=250.,
                        N_r=2000, N_bc=400, N_ic=400, n_epochs=8000,
                        lr=1e-3, batch_r=500, cache_freq=500, verbose=True):
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    model  = PINN_BS(4, 64).to(DEVICE)
    opt    = optim.Adam(model.parameters(), lr=lr)
    sched  = optim.lr_scheduler.ExponentialLR(opt, gamma=0.9997)
    pts    = dat.amostrar_colocation(N_r, N_bc, N_ic, K, S_max, T, r, DEVICE)
    hist   = {"total":[], "res":[], "ic":[], "bc":[]}
    n_b_r  = max(1, (N_r+batch_r-1)//batch_r)

    for ep in range(n_epochs):
        model.train(); opt.zero_grad()
        L_r_acc = 0.0
        for i in range(0, N_r, batch_r):
            res_b = bs_residual_pinn(model, pts["S_r"][i:i+batch_r],
                                      pts["tau_r"][i:i+batch_r], K, r, sigma_true, S_max, T)
            L_b = (res_b**2).mean()/n_b_r
            L_b.backward(); L_r_acc += L_b.detach().item()
        S_n,t_n = normalizar(pts["S_ic"],  pts["tau_ic"],  K, S_max, T)
        L_ic  = ((model(S_n,t_n)*K - pts["V_ic"])**2).mean()
        S_n,t_n = normalizar(pts["S_bc0"], pts["tau_bc0"], K, S_max, T)
        L_bc0 = ((model(S_n,t_n)*K - pts["V_bc0"])**2).mean()
        S_n,t_n = normalizar(pts["S_bcS"], pts["tau_bcS"], K, S_max, T)
        L_bcS = ((model(S_n,t_n)*K - pts["V_bcS"])**2).mean()
        L_bc_ic = 10.*L_ic + 5.*(L_bc0+L_bcS)
        L_bc_ic.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        L_tot = L_r_acc + L_bc_ic.item()
        hist["total"].append(L_tot); hist["res"].append(L_r_acc)
        hist["ic"].append(L_ic.item()); hist["bc"].append((L_bc0+L_bcS).item())
        if DEVICE.type=="cuda" and ep%cache_freq==0 and ep>0: torch.cuda.empty_cache()
        if verbose and (ep%2000==0 or ep==n_epochs-1):
            mem = f" | {gpu_mem_stats()}" if DEVICE.type=="cuda" else ""
            print(f"  ep {ep:5d} | L={L_tot:.2e} | L_res={L_r_acc:.2e} | L_ic={L_ic.item():.2e}{mem}")
    if DEVICE.type=="cuda": torch.cuda.empty_cache()
    return model, hist


print("Treinando PINN (problema direto, σ=0.20)...")
pinn_dir, hist_dir = treinar_pinn_direto(sigma_true=0.20, n_epochs=8000, verbose=True)

Treinando PINN (problema direto, σ=0.20)...


/home/mateus25032/miniconda3/lib/python3.13/site-packages/torch/autograd/graph.py:841: UserWarning:

Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)



  ep     0 | L=1.27e+05 | L_res=8.32e+00 | L_ic=3.20e+03 | VRAM 17/24 MB
  ep  2000 | L=7.90e+01 | L_res=6.19e+00 | L_ic=5.64e+00 | VRAM 17/24 MB
  ep  4000 | L=2.08e+01 | L_res=1.74e+00 | L_ic=1.16e+00 | VRAM 17/24 MB
  ep  6000 | L=9.66e+00 | L_res=6.58e-01 | L_ic=7.90e-01 | VRAM 17/24 MB
  ep  7999 | L=7.13e+00 | L_res=3.42e-01 | L_ic=6.23e-01 | VRAM 17/24 MB


In [ ]:
# ── Validação e curva de aprendizado ─────────────────────────────────────────
def avaliar_pinn(model, S_vals, tau_val, K=100., S_max=250., T=1.0):
    model.eval()
    with torch.no_grad():
        S_t = torch.tensor(S_vals, dtype=torch.float32).reshape(-1,1).to(DEVICE)
        tau_t = torch.full_like(S_t, tau_val)
        S_n, tau_n = normalizar(S_t, tau_t, K, S_max, T)
        return model(S_n, tau_n).cpu().numpy().flatten()*K

fig_lc = plt_sde.fig_loss_curve(hist_dir, "<b>Figura 7a</b> — Curva de Aprendizado: PINN Direta")
fig_lc.show()

S_ev = np.linspace(50, 200, 150)
V_pinn_dict = {tau_v: avaliar_pinn(pinn_dir, S_ev, tau_v) for tau_v in [0.25,0.5,1.0]}
V_an_dict   = {tau_v: bs.bs_call(S_ev,100.,0.05,0.20,tau_v) for tau_v in [0.25,0.5,1.0]}

fig_val = plt_sde.fig_pinn_validacao(S_ev, [0.25,0.5,1.0], V_pinn_dict, V_an_dict)
fig_val.show()

A curva de aprendizado (primeira figura) exibe a hierarquia temporal característica das PINNs [5]: a perda de condição inicial $\mathcal{L}_{ic}$ decresce rapidamente nas primeiras épocas, pois satisfazer condições pontuais é um problema de otimização localizado; o resíduo da EDP $\mathcal{L}_{res}$ converge mais lentamente, porque impor a física em pontos de colocação distribuídos por todo o domínio exige que a rede aprenda a curvatura global da solução. O terceiro painel, com as perdas de condição de contorno $\mathcal{L}_{bc}$, confirma que as condições em $S=0$ e $S\to S_\max$ são satisfeitas cedo no treinamento e permanecem com valores reduzidos ao final — evidência de que a PINN respeita simultaneamente as restrições físicas laterais e o interior do domínio. A segunda figura valida a solução aprendida contra a fórmula analítica para três maturidades: o acordo visual é praticamente perfeito em toda a faixa $S\in[50,200]$, demonstrando que a rede generalizou a solução da EDP globalmente, e não apenas nos pontos de colocação.

## 4.2 PINN para o Problema Inverso
<a id="sec42"></a>

No problema direto, $\sigma$ é conhecido e a PINN aprende $V(S,\tau;\theta)$. No problema inverso, $\sigma$ é desconhecido e deve ser inferido a partir de observações $\{(S_i, K_i, \tau_i, C_i^{\text{obs}})\}$ de preços de opções. A abordagem mais natural é tratar $\sigma$ como um parâmetro treinável adicional do modelo. A perda de dados $\mathcal{L}_{\text{dados}} = \frac{1}{N_d}\sum_i (\hat{V}(S_i,\tau_i;\theta,\sigma) - C_i^{\text{obs}})^2$ incentiva $\sigma$ a explicar os preços observados, enquanto o resíduo da EDP $\mathcal{L}_{\text{res}}$ garante que toda a superfície de preços seja globalmente consistente com Black-Scholes.

A diferença crucial em relação à inversão de Newton-Raphson é que, enquanto NR inverte ponto a ponto (um $\sigma$ por observação), a PINN inversa estima um único $\sigma$ que minimiza o desajuste em todos os pontos simultaneamente, regularizado pela física da EDP. Isso torna a estimativa muito mais robusta a ruído e _outliers_ isolados. Contudo, essa estimativa ainda é pontual — não quantifica a incerteza. É esse salto que a B-PINN realiza.

<small>[↑ Voltar ao topo](#toc)</small>

## 4.3 Análise de Hadamard e Ill-posedness
<a id="sec43"></a>

Jacques Hadamard definiu em 1902 que um problema é bem-posto se satisfaz três condições: existência de solução, unicidade da solução e dependência contínua da solução nos dados (estabilidade). Um problema que falha em qualquer dessas condições é mal-posto (_ill-posed_). [[6]](#referencias)

Para o problema de recuperar $\sigma$ a partir de preços de opções, existência e unicidade estão garantidas pelo Vega positivo (como discutido na Seção 2.3). Porém, a **estabilidade** falha quando os dados são ruidosos. Para ver isso, considere uma perturbação $\delta C$ no preço observado. A perturbação resultante em $\sigma^*$ é, pela regra da derivação implícita:
$$\delta\sigma^* \approx \frac{\delta C}{\mathcal{V}(\sigma^*)}.$$
Quando $\mathcal{V} \to 0$ (opções profundamente ITM ou OTM, ou de maturidade muito curta), uma perturbação $\delta C$ aparentemente pequena produz um erro $\delta\sigma^*$ ilimitado. Isso é o **ill-conditioning** do problema inverso, que manifesta-se como o comportamento errático da IV em opções fora do dinheiro observado na Figura 4.

A abordagem Bayesiana não elimina o _ill-posedness_ — ela o quantifica. Um posterior amplo de $\sigma$ em regiões de baixo Vega comunica que os dados disponíveis não são informativos o suficiente para determinar $\sigma$ com precisão. Isso é a resposta correta, não uma falha do método.

<small>[↑ Voltar ao topo](#toc)</small>

## 4.4 Redes Neurais Bayesianas — Inferência Variacional
<a id="sec44"></a>

Uma rede neural Bayesiana trata os pesos como variáveis aleatórias ao invés de parâmetros determinísticos. Dado um prior $p(\theta)$ e uma verossimilhança $p(\mathcal{D}\mid\theta)$, o objetivo é calcular o posterior:
$$
p(\theta \mid \mathcal{D}) = \frac{p(\mathcal{D}\mid\theta)\,p(\theta)}{p(\mathcal{D})},
$$

onde $p(\mathcal{D}) = \int p(\mathcal{D}\mid\theta)p(\theta)\,d\theta$ é intratável para redes neurais de qualquer tamanho. A inferência variacional contorna isso aproximando o posterior por uma distribuição mais simples $q_\phi(\theta)$ (parametrizada por $\phi$), encontrada minimizando $\text{KL}[q_\phi \| p(\cdot\mid\mathcal{D})]$. Isso equivale a maximizar o ELBO (Evidence Lower BOund):
$$
\mathcal{L}_{\text{ELBO}} = \underbrace{\mathbb{E}_{q_\phi}[\log p(\mathcal{D}\mid\theta)]}_{\text{Ajuste aos Dados}} - \underbrace{\text{KL}[q_\phi(\theta) \| p(\theta)]}_{\text{Regularização Bayesiana}}
$$

O método Bayes by Backprop [[7]](#referencias) torna isso computacionalmente tratável usando um campo médio gaussiano $q_\phi(w) = \mathcal{N}(\mu_w, \sigma_w^2)$ com $\sigma_w = \text{softplus}(\rho_w)$ para garantir positividade, e o truque da reparametrização $w = \mu_w + \sigma_w \cdot \varepsilon$, $\varepsilon \sim \mathcal{N}(0,1)$, para obter gradientes sem viés em relação a $\phi$. A divergência KL entre duas gaussianas tem forma analítica fechada:
$$\text{KL}[\mathcal{N}(\mu,\sigma^2) \| \mathcal{N}(0,p^2)] = \log\frac{p}{\sigma} + \frac{\sigma^2+\mu^2}{2p^2} - \frac{1}{2}$$

<small>[↑ Voltar ao topo](#toc)</small>

In [ ]:
# ════════════════════════════════════════════════════════════
# BACKBONE 3 — Camada Linear Bayesiana (Bayes by Backprop)
# ════════════════════════════════════════════════════════════

class BayesLinear(nn.Module):
    """
    Camada linear com pesos variacionais w ~ N(mu, softplus(rho)²).
    Prior: w ~ N(0, prior_sigma²).
    KL: forma analítica fechada entre duas gaussianas.
    """
    def __init__(self, in_f, out_f, prior_sigma=1.0):
        super().__init__()
        self.ps   = prior_sigma
        self.w_mu  = nn.Parameter(torch.zeros(out_f, in_f))
        self.w_rho = nn.Parameter(torch.full((out_f, in_f), -3.0))
        self.b_mu  = nn.Parameter(torch.zeros(out_f))
        self.b_rho = nn.Parameter(torch.full((out_f,), -3.0))
        nn.init.xavier_normal_(self.w_mu)

    def forward(self, x):
        ws = torch.log1p(torch.exp(self.w_rho))  # softplus → positivo
        bs = torch.log1p(torch.exp(self.b_rho))
        if self.training:                          # reparametrização
            w = self.w_mu + ws*torch.randn_like(self.w_mu)
            b = self.b_mu + bs*torch.randn_like(self.b_mu)
        else:                                      # média na inferência
            w, b = self.w_mu, self.b_mu
        return nn.functional.linear(x, w, b)

    def kl_div(self):
        """KL[q(w) || p(w)] — forma analítica gaussiana."""
        ps = self.ps
        ws = torch.log1p(torch.exp(self.w_rho))
        bs = torch.log1p(torch.exp(self.b_rho))
        kw = (torch.log(ps/ws) + (ws**2+self.w_mu**2)/(2*ps**2) - 0.5).sum()
        kb = (torch.log(ps/bs) + (bs**2+self.b_mu**2)/(2*ps**2) - 0.5).sum()
        return kw + kb


bl = BayesLinear(2, 4).train()
x_demo = torch.randn(5, 2)
o1, o2 = bl(x_demo), bl(x_demo)
print(f"✓ BayesLinear OK")
print(f"  Saídas distintas (amostragem): {not torch.allclose(o1,o2)}")
print(f"  KL inicial: {bl.kl_div().item():.3f}")

✓ BayesLinear OK
  Saídas distintas (amostragem): True
  KL inicial: 31.780


## 4.5 Bayesian PINN — Unificação
<a id="sec45"></a>

A Bayesian PINN unifica dois _priors_ estruturais complementares:
1. _Prior_ físico: é a EDP de Black-Scholes: o resíduo $\mathcal{L}_{\text{res}}$ atua como um potencial de energia que penaliza configurações de $(\theta, \sigma)$ inconsistentes com a física do modelo.
2. _Prior_ epistêmico: é a distribuição gaussiana variacional sobre os pesos e sobre $\sigma$: ele regulariza a estimativa e permite propagar a incerteza dos dados até o posterior de $\sigma$.

Para $\sigma$, adotamos um _prior_ log-normal centrado em valores típicos de mercado: $\log\sigma \sim \mathcal{N}(-1.6,\, 0.5^2)$, que corresponde a $\mathbb{E}[\sigma] \approx 0.20$ com suporte concentrado em $[0.08, 0.50]$. Após o treinamento, amostramos o posterior $\{\sigma^{(k)}\}_{k=1}^K$ via `sample_sigma()` no modo `train()`, obtendo uma distribuição completa de incerteza sobre a volatilidade — não apenas um ponto estimado.

A perda total da B-PINN combina _likelihood_, física e regularização Bayesiana:
$$
\mathcal{L}_{\text{B-PINN}} = \underbrace{\lambda_d\mathcal{L}_{\text{dados}} + \lambda_r\mathcal{L}_{\text{res}} + \lambda_{bc}\mathcal{L}_{BC}}_{\text{Likelihood}} + \beta\,\underbrace{(\text{KL}_{\text{redes}} + \gamma\,\text{KL}_{\sigma})}_{\text{Regularizador ELBO}}
$$

<small>[↑ Voltar ao topo](#toc)</small>

In [ ]:
# ════════════════════════════════════════════════════════════
# BACKBONE 4 — Bayesian PINN com σ como variável latente
# ════════════════════════════════════════════════════════════
import math

class BayesianPINN(nn.Module):
    """
    Bayesian PINN para Black-Scholes.
    Pesos: distribuição variacional gaussiana (BayesLinear).
    σ: log-normal variacional com prior log N(-1.6, 0.5²).
    Posterior de σ obtido por amostragem em modo train().
    """
    def __init__(self, hidden_layers=3, hidden_dim=48, prior_sigma=1.0, sigma_init=0.30):
        super().__init__()
        sizes = [2] + [hidden_dim]*hidden_layers + [1]
        self.layers = nn.ModuleList([
            BayesLinear(sizes[j], sizes[j+1], prior_sigma) for j in range(len(sizes)-1)])
        self.act = nn.Tanh()
        self.log_sig_mu  = nn.Parameter(torch.tensor(math.log(sigma_init)))
        self.log_sig_rho = nn.Parameter(torch.tensor(-3.0))

    @property
    def sigma(self):
        return torch.exp(self.log_sig_mu)

    def sample_sigma(self):
        """Amostra σ via reparametrização (modo train) ou retorna a média (modo eval)."""
        if self.training:
            std = torch.log1p(torch.exp(self.log_sig_rho))
            return torch.exp(self.log_sig_mu + std*torch.randn(1, device=self.log_sig_mu.device))
        return self.sigma

    def forward(self, S_n, tau_n):
        x = torch.cat([S_n, tau_n], dim=1)
        for j, layer in enumerate(self.layers):
            x = layer(x)
            if j < len(self.layers)-1: x = self.act(x)
        return x

    def kl_redes(self):
        return sum(l.kl_div() for l in self.layers)

    def kl_sigma(self):
        """KL[q(σ) || p(σ)] com prior log N(-1.6, 0.5²)."""
        std = torch.log1p(torch.exp(self.log_sig_rho))
        pm  = torch.tensor(-1.6, device=self.log_sig_mu.device)
        ps  = torch.tensor(0.5,  device=self.log_sig_mu.device)
        return torch.log(ps/std) + (std**2+(self.log_sig_mu-pm)**2)/(2*ps**2) - 0.5


bpinn_t = BayesianPINN().to(DEVICE)
x_t = torch.rand(8,1,device=DEVICE)
print(f"✓ BayesianPINN | saída shape={bpinn_t(x_t,x_t).shape}")
print(f"  σ inicial: {bpinn_t.sigma.item():.3f} | KL redes: {bpinn_t.kl_redes().item():.2f}")

✓ BayesianPINN | saída shape=torch.Size([8, 1])
  σ inicial: 0.300 | KL redes: 12419.92


In [ ]:
# ════════════════════════════════════════════════════════════
# BACKBONE 5 — Laço de treinamento da B-PINN
# ════════════════════════════════════════════════════════════

def treinar_bpinn(sigma_true, eta=0.03, N_d=80, K=100., r=0.05, T=1.5,
                  S_max=250., N_r=2000, N_bc=300, N_ic=300,
                  n_epochs=10000, lr=5e-4, beta_kl=0.01,
                  batch_r=400, cache_freq=500, verbose=True, seed=SEED):
    if DEVICE.type=="cuda": torch.cuda.empty_cache()
    dados = dat.gerar_dados_opcoes(sigma_true, N=N_d, eta=eta, seed=seed)
    S_d   = torch.tensor(dados["S"],    dtype=torch.float32).reshape(-1,1).to(DEVICE)
    tau_d = torch.tensor(dados["tau"],  dtype=torch.float32).reshape(-1,1).to(DEVICE)
    C_d   = torch.tensor(dados["C_obs"],dtype=torch.float32).reshape(-1,1).to(DEVICE)
    pts   = dat.amostrar_colocation(N_r,N_bc,N_ic,K,S_max,T,r,DEVICE,seed=seed)
    model = BayesianPINN(3,48,sigma_init=0.30).to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt,T_max=n_epochs,eta_min=lr*0.01)
    hist  = {"total":[],"sigma":[],"kl":[],"l_data":[],"l_res":[]}
    n_b_r = max(1,(N_r+batch_r-1)//batch_r)

    for ep in range(n_epochs):
        model.train(); opt.zero_grad()
        sig_val = model.sample_sigma().item()
        S_n,t_n = normalizar(S_d,tau_d,K,S_max,T)
        L_data  = ((model(S_n,t_n)*K - C_d)**2).mean()
        L_r_acc = 0.0
        for i in range(0,N_r,batch_r):
            res_b = bs_residual_pinn(model,pts["S_r"][i:i+batch_r],
                                      pts["tau_r"][i:i+batch_r],K,r,sig_val,S_max,T)
            L_b = (res_b**2).mean()/n_b_r
            L_b.backward(); L_r_acc += L_b.detach().item()
        S_n,t_n = normalizar(pts["S_ic"], pts["tau_ic"],  K,S_max,T)
        L_ic  = ((model(S_n,t_n)*K - pts["V_ic"])**2).mean()
        S_n,t_n = normalizar(pts["S_bc0"],pts["tau_bc0"],K,S_max,T)
        L_bc0 = ((model(S_n,t_n)*K - pts["V_bc0"])**2).mean()
        S_n,t_n = normalizar(pts["S_bcS"],pts["tau_bcS"],K,S_max,T)
        L_bcS = ((model(S_n,t_n)*K - pts["V_bcS"])**2).mean()
        kl    = model.kl_redes() + 10.*model.kl_sigma()
        L_rest= 20.*L_data+10.*L_ic+5.*(L_bc0+L_bcS)+beta_kl*kl
        L_rest.backward()
        nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step(); sched.step()
        L_tot = L_r_acc + L_rest.item()
        hist["total"].append(L_tot); hist["sigma"].append(model.sigma.item())
        hist["kl"].append(kl.item()); hist["l_data"].append(L_data.item())
        hist["l_res"].append(L_r_acc)
        if DEVICE.type=="cuda" and ep%cache_freq==0 and ep>0: torch.cuda.empty_cache()
        if verbose and (ep%2000==0 or ep==n_epochs-1):
            mem = f" | {gpu_mem_stats()}" if DEVICE.type=="cuda" else ""
            print(f"  ep {ep:5d} | loss={L_tot:.3e} | σ={model.sigma.item():.4f} | σ_true={sigma_true:.4f}{mem}")
    if DEVICE.type=="cuda": torch.cuda.empty_cache()
    return model, hist, dados


SIGMA_TRUE = 0.25
print(f"Treinando B-PINN | σ_true={SIGMA_TRUE}, ruído=3%, N=80...")
bpinn_m, bpinn_h, bpinn_d = treinar_bpinn(SIGMA_TRUE, eta=0.03, N_d=80, n_epochs=10000, verbose=True)

Treinando B-PINN | σ_true=0.25, ruído=3%, N=80...
  ep     0 | loss=3.228e+05 | σ=0.2999 | σ_true=0.2500 | VRAM 18/24 MB
  ep  2000 | loss=6.735e+03 | σ=0.2021 | σ_true=0.2500 | VRAM 18/24 MB
  ep  4000 | loss=9.016e+03 | σ=0.2019 | σ_true=0.2500 | VRAM 18/24 MB
  ep  6000 | loss=5.019e+03 | σ=0.2019 | σ_true=0.2500 | VRAM 18/24 MB
  ep  8000 | loss=1.473e+03 | σ=0.2019 | σ_true=0.2500 | VRAM 18/24 MB
  ep  9999 | loss=1.015e+04 | σ=0.2019 | σ_true=0.2500 | VRAM 18/24 MB


In [ ]:
# ── Posterior de σ e visualizações ──────────────────────────────────────────
bpinn_m.train()
with torch.no_grad():
    sigma_post = np.array([bpinn_m.sample_sigma().item() for _ in range(2000)])
bpinn_m.eval()

resumo = met.resumo_posterior(sigma_post, sigma_true=SIGMA_TRUE)
ic95   = resumo["ic_95"]
print(f"✓ Posterior de σ:")
print(f"  Média  : {resumo['media']:.4f}  (verdadeiro: {SIGMA_TRUE})")
print(f"  Desvio : {resumo['desvio']:.4f}")
print(f"  Moda   : {resumo['moda_kde']:.4f}")
print(f"  IC 95% : [{ic95['baixo']:.4f}, {ic95['alto']:.4f}]")
print(f"  Viés   : {resumo['vies']:.4f}")
print(f"  Cobertura IC95%: {'✓ Sim' if resumo['cobertura_95'] else '✗ Não'}")

fig1 = plt_sde.fig_bpinn_convergencia(bpinn_h, SIGMA_TRUE)
fig1.show()

fig2 = plt_sde.fig_posterior_sigma(sigma_post, SIGMA_TRUE,
    "Posterior de σ | B-PINN | σ_true=0.25, η=3%")
fig2.show()

✓ Posterior de σ:
  Média  : 0.2226  (verdadeiro: 0.25)
  Desvio : 0.0932
  Moda   : 0.1853
  IC 95% : [0.0949, 0.4510]
  Viés   : 0.0274
  Cobertura IC95%: ✓ Sim


A figura de convergência (primeira) organiza o diagnóstico do treinamento Bayesiano em três painéis. O painel de perdas mostra a evolução conjunta dos termos de likelihood e da regularização KL: as oscilações iniciais — mais intensas do que na PINN clássica — refletem a amostragem estocástica dos pesos e de $\sigma$ em cada passo de gradiente, própria do estimador de Monte Carlo no ELBO. O painel central registra a trajetória de $\mu_\sigma$ (a média da distribuição variacional de $\log\sigma$) ao longo das épocas: partindo do valor inicial $\sigma_0 = 0.30$, o parâmetro é atraído em direção a $\sigma_\text{true}$ à medida que a likelihood supera o prior — o saldo dinâmico entre ajuste aos dados e regularização Bayesiana. O terceiro painel exibe a divergência KL entre o posterior variacional e o prior log-normal $\mathcal{N}(-1.6, 0.5^2)$: seu valor residual ao final do treinamento quantifica o quanto os dados foram informativos para atualizar a crença inicial — uma KL alta indica posterior próximo do prior, sinalizando dados pouco discriminativos [7]. A segunda figura mostra o posterior amostrado de $\sigma$: sua largura expressa a incerteza epistêmica residual após incorporar todos os dados e impor a física da EDP, e a cobertura do IC 95% é o critério central de calibração do método.

---
# 5. Experimentos: A Questão Central em Prática
<a id="parte5"></a>

> *"Até que ponto a volatilidade implícita é identificável a partir de dados ruidosos sob restrições físicas?"*

Esta parte transforma a questão central numa série de experimentos quantitativos controlados. Variamos sistematicamente o nível de ruído $\eta$ e o número de observações $N$, medindo a qualidade do posterior de $\sigma$ via três métricas: viés, largura do IC 95% e cobertura empírica. O experimento com dados reais fecha a análise com um teste em condições de mercado.

<small>[↑ Voltar ao topo](#toc)</small>

## 5.1 Protocolo Experimental e Análise de Ruído
<a id="sec51"></a>

A estratégia experimental é a seguinte. Para cada combinação $(\eta, N)$, gera-se um _dataset_ sintético com $\sigma_{\text{true}} = 0.25$, treina-se a B-PINN, coleta-se 1000 amostras do _posterior_ de $\sigma$ e então calcula-se as métricas via `src_SDE.metrics.resumo_posterior`. O controle total sobre $\sigma_{\text{true}}$ (impossível com dados reais) é o que permite avaliar a calibração do _posterior_ — se as afirmações probabilísticas da B-PINN correspondem às frequências empíricas.

In [ ]:
def treinar_bpinn_rapido(sigma_true, eta=0.03, N_d=60, K=100., r=0.05,
                          T=1.5, S_max=250., n_epochs=6000, lr=5e-4,
                          batch_r=400, seed=42):
    """Versão compacta — retorna resumo do posterior via src_SDE.metrics."""
    if DEVICE.type=="cuda": torch.cuda.empty_cache()
    dados = dat.gerar_dados_opcoes(sigma_true,N=N_d,eta=eta,seed=seed)
    S_d  = torch.tensor(dados["S"],    dtype=torch.float32).reshape(-1,1).to(DEVICE)
    Td   = torch.tensor(dados["tau"],  dtype=torch.float32).reshape(-1,1).to(DEVICE)
    C_d  = torch.tensor(dados["C_obs"],dtype=torch.float32).reshape(-1,1).to(DEVICE)
    N_r  = 1500
    pts  = dat.amostrar_colocation(N_r,200,200,K,S_max,T,r,DEVICE,seed=seed)
    model= BayesianPINN(3,40,sigma_init=0.30).to(DEVICE)
    opt  = optim.Adam(model.parameters(),lr=lr)
    sched= optim.lr_scheduler.CosineAnnealingLR(opt,T_max=n_epochs,eta_min=lr*0.01)
    n_b_r= max(1,(N_r+batch_r-1)//batch_r)
    for ep in range(n_epochs):
        model.train(); opt.zero_grad()
        sig_val = model.sample_sigma().item()
        S_n,t_n = normalizar(S_d,Td,K,S_max,T)
        L_data  = ((model(S_n,t_n)*K - C_d)**2).mean()
        L_r_acc = 0.
        for i in range(0,N_r,batch_r):
            res_b = bs_residual_pinn(model,pts["S_r"][i:i+batch_r],
                                      pts["tau_r"][i:i+batch_r],K,r,sig_val,S_max,T)
            L_b=(res_b**2).mean()/n_b_r; L_b.backward(); L_r_acc+=L_b.item()
        S_n,t_n=normalizar(pts["S_ic"],pts["tau_ic"],K,S_max,T)
        L_ic=((model(S_n,t_n)*K-pts["V_ic"])**2).mean()
        S_n,t_n=normalizar(pts["S_bc0"],pts["tau_bc0"],K,S_max,T)
        L_bc0=((model(S_n,t_n)*K-pts["V_bc0"])**2).mean()
        S_n,t_n=normalizar(pts["S_bcS"],pts["tau_bcS"],K,S_max,T)
        L_bcS=((model(S_n,t_n)*K-pts["V_bcS"])**2).mean()
        kl=model.kl_redes()+10.*model.kl_sigma()
        L_rest=20.*L_data+10.*L_ic+5.*(L_bc0+L_bcS)+0.01*kl
        L_rest.backward()
        nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step(); sched.step()
        if DEVICE.type=="cuda" and ep%500==0 and ep>0: torch.cuda.empty_cache()
    model.train()
    with torch.no_grad(): sp=np.array([model.sample_sigma().item() for _ in range(1000)])
    if DEVICE.type=="cuda": torch.cuda.empty_cache()
    return met.resumo_posterior(sp, sigma_true=sigma_true)


print("✓ treinar_bpinn_rapido definido (com mini-batch de resíduo).")

✓ treinar_bpinn_rapido definido (com mini-batch de resíduo).


In [ ]:
# ── Experimento 1: σ vs. nível de ruído η ────────────────────────────────────
noise_vals = [0.0, 0.01, 0.03, 0.05, 0.10]
SIG_EXP = 0.25; N_EXP = 80
res_ruido = []

print(f"Experimento 1: σ_true={SIG_EXP}, N={N_EXP} observações")
print("-"*65)
for eta in noise_vals:
    r_ = treinar_bpinn_rapido(SIG_EXP, eta=eta, N_d=N_EXP, n_epochs=6000)
    res_ruido.append(r_)
    ic = r_["ic_95"]; cob = "✓" if r_.get("cobertura_95",0) else "✗"
    print(f"  η={eta*100:5.1f}% | σ̂={r_['media']:.4f}±{r_['desvio']:.4f}"
          f" | IC=[{ic['baixo']:.3f},{ic['alto']:.3f}]"
          f" | viés={r_['vies']:.4f} | {cob}")

fig = plt_sde.fig_identificabilidade_ruido(noise_vals, res_ruido, SIG_EXP)
fig.show()

Experimento 1: σ_true=0.25, N=80 observações
-----------------------------------------------------------------
  η=  0.0% | σ̂=0.2069±0.0413 | IC=[0.137,0.300] | viés=0.0431 | ✓
  η=  1.0% | σ̂=0.2075±0.0426 | IC=[0.140,0.305] | viés=0.0425 | ✓
  η=  3.0% | σ̂=0.2068±0.0410 | IC=[0.138,0.293] | viés=0.0432 | ✓
  η=  5.0% | σ̂=0.2042±0.0397 | IC=[0.139,0.290] | viés=0.0458 | ✓
  η= 10.0% | σ̂=0.2061±0.0436 | IC=[0.133,0.305] | viés=0.0439 | ✓


O painel esquerdo exibe as estimativas $\hat\sigma$ com seus intervalos de confiança de 95% em função de $\eta$: mesmo sob ruído elevado, a média posterior permanece próxima a $\sigma_\text{true}$, evidenciando que o resíduo da EDP ancora a estimativa globalmente, impedindo a deriva que ocorreria com inversão ponto a ponto. O painel central exibe a largura do IC 95%, que cresce monotonicamente com $\eta$: esse alargamento não é uma falha, mas a resposta correta do posterior Bayesiano à perda de informação — um estimador bem calibrado deve ser mais incerto quando os dados são mais ruidosos. O painel direito registra o viés e a cobertura empírica do IC 95%: a cobertura próxima a 95% é o critério de calibração formal que distingue a B-PINN de métodos que minimizam apenas o erro pontual, pois garante que as afirmações probabilísticas do posterior correspondam às frequências observadas na prática [5][6][10].

In [ ]:
# ── Posteriors sobrepostos por nível de ruído ─────────────────────────────────
def amostrar_posterior(sigma_true, eta, N_d, n_epochs=5000, seed=42):
    """Retorna array de amostras do posterior de σ."""
    dados=dat.gerar_dados_opcoes(sigma_true,N=N_d,eta=eta,seed=seed)
    S_d=torch.tensor(dados["S"],dtype=torch.float32).reshape(-1,1).to(DEVICE)
    Td =torch.tensor(dados["tau"],dtype=torch.float32).reshape(-1,1).to(DEVICE)
    C_d=torch.tensor(dados["C_obs"],dtype=torch.float32).reshape(-1,1).to(DEVICE)
    N_r=1500; pts=dat.amostrar_colocation(N_r,200,200,100.,250.,1.5,0.05,DEVICE,seed=seed)
    model=BayesianPINN(3,40,sigma_init=0.30).to(DEVICE)
    opt=optim.Adam(model.parameters(),lr=5e-4)
    sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=n_epochs,eta_min=5e-6)
    n_b_r=max(1,(N_r+400-1)//400)
    for ep in range(n_epochs):
        model.train(); opt.zero_grad()
        sv=model.sample_sigma().item()
        S_n,t_n=normalizar(S_d,Td,100.,250.,1.5)
        L_data=((model(S_n,t_n)*100.-C_d)**2).mean()
        L_r_acc=0.
        for i in range(0,N_r,400):
            res_b=bs_residual_pinn(model,pts["S_r"][i:i+400],pts["tau_r"][i:i+400],100.,0.05,sv,250.,1.5)
            L_b=(res_b**2).mean()/n_b_r; L_b.backward(); L_r_acc+=L_b.item()
        S_n,t_n=normalizar(pts["S_ic"],pts["tau_ic"],100.,250.,1.5)
        L_ic=((model(S_n,t_n)*100.-pts["V_ic"])**2).mean()
        S_n,t_n=normalizar(pts["S_bc0"],pts["tau_bc0"],100.,250.,1.5)
        L_bc0=((model(S_n,t_n)*100.-pts["V_bc0"])**2).mean()
        S_n,t_n=normalizar(pts["S_bcS"],pts["tau_bcS"],100.,250.,1.5)
        L_bcS=((model(S_n,t_n)*100.-pts["V_bcS"])**2).mean()
        kl=model.kl_redes()+10.*model.kl_sigma()
        L_rest=20.*L_data+10.*L_ic+5.*(L_bc0+L_bcS)+0.01*kl
        L_rest.backward()
        nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step(); sched.step()
        if DEVICE.type=="cuda" and ep%500==0 and ep>0: torch.cuda.empty_cache()
    model.train()
    with torch.no_grad(): sp=np.array([model.sample_sigma().item() for _ in range(800)])
    return sp


print("Coletando posteriors sobrepostos...")
posteriors_eta = {}
for eta in noise_vals:
    sp = amostrar_posterior(SIG_EXP, eta, N_EXP, n_epochs=5000)
    posteriors_eta[f"η={eta*100:.0f}%"] = sp
    print(f"  η={eta*100:.0f}%: σ̂={sp.mean():.4f} ± {sp.std():.4f}")

fig = plt_sde.fig_posterior_multiplo(posteriors_eta, SIG_EXP)
fig.show()

Coletando posteriors sobrepostos...
  η=0%: σ̂=0.2030 ± 0.0336
  η=1%: σ̂=0.2061 ± 0.0330
  η=3%: σ̂=0.2060 ± 0.0363
  η=5%: σ̂=0.2042 ± 0.0321
  η=10%: σ̂=0.2022 ± 0.0314


A sobreposição das densidades posteriores em função de $\eta$ ilustra o mecanismo Bayesiano de atualização de crenças: para $\eta = 0$, a likelihood é fortemente informativa e o posterior é estreito, concentrado sobre $\sigma_\text{true}$; à medida que $\eta$ cresce, o posterior alarga-se e desloca-se em direção ao prior log-normal, pois dados menos informativos oferecem menos evidência para afastar a distribuição inicial. Crucialmente, as modas permanecem próximas a $\sigma_\text{true}$ mesmo sob ruído moderado — evidência de que o resíduo da EDP ancora a estimativa mesmo quando cada observação individual é pouco discriminativa. Essa propriedade distingue a B-PINN de estimadores pontuais que simplesmente degradam silenciosamente com dados ruidosos: aqui, a incerteza aumenta de forma visível e quantificada, permitindo ao usuário reconhecer quando os dados são insuficientes para identificar $\sigma$ com precisão [10].

## 5.2 Comparação: Newton-Raphson vs. B-PINN
<a id="sec52"></a>

A comparação a seguir coloca frente a frente a inversão clássica ponto a ponto (Newton-Raphson, que fornece uma IV por observação) e o _posterior_ B-PINN (que fornece uma distribuição completa de $\sigma$). É importante notar que a comparação não é diretamente entre dois pontos estimados, mas entre uma coleção de pontos (IVs individuais do NR) e uma distribuição (_posterior_ B-PINN): o primeiro não é um _posterior_ válido, enquanto o segundo é.

<small>[↑ Voltar ao topo](#toc)</small>

In [ ]:
# ── Comparação NR vs. B-PINN ─────────────────────────────────────────────────
sig_comp, eta_comp, N_comp = 0.25, 0.04, 80
d_comp = dat.gerar_dados_opcoes(sig_comp, N=N_comp, eta=eta_comp, seed=42)

ivs_nr = np.array([bs.iv_newton(c,s,k,0.05,t)
    for c,s,k,t in zip(d_comp["C_obs"],d_comp["S"],d_comp["K"],d_comp["tau"])])
ivs_nr = ivs_nr[~np.isnan(ivs_nr) & (ivs_nr>0.01) & (ivs_nr<3.)]

sp_comp = amostrar_posterior(sig_comp, eta_comp, N_comp, n_epochs=5000)
res_nr  = met.resumo_posterior(ivs_nr,  sigma_true=sig_comp)
res_bp  = met.resumo_posterior(sp_comp, sigma_true=sig_comp)

print("="*65)
print(f"COMPARAÇÃO | σ_true={sig_comp}, η={eta_comp*100:.0f}%, N={N_comp}")
print("="*65)
print(f"Newton-Raphson  | σ̂={res_nr['media']:.4f} ± {res_nr['desvio']:.4f} | viés={res_nr['vies']:.4f}")
print(f"B-PINN posterior| σ̂={res_bp['media']:.4f} ± {res_bp['desvio']:.4f} | viés={res_bp['vies']:.4f}")
print(f"  IC95% B-PINN: [{res_bp['ic_95']['baixo']:.4f}, {res_bp['ic_95']['alto']:.4f}]")
print(f"  Cobertura IC95%: {'✓ Sim' if res_bp['cobertura_95'] else '✗ Não'}")

fig = plt_sde.fig_comparacao_metodos(ivs_nr, sp_comp, sig_comp)
fig.show()

COMPARAÇÃO | σ_true=0.25, η=4%, N=80
Newton-Raphson  | σ̂=0.2503 ± 0.0256 | viés=0.0003
B-PINN posterior| σ̂=0.2063 ± 0.0353 | viés=0.0437
  IC95% B-PINN: [0.1469, 0.2823]
  Cobertura IC95%: ✓ Sim


O painel esquerdo exibe a distribuição das volatilidades implícitas obtidas pelo Newton–Raphson — uma estimativa por observação: a dispersão reflete tanto o ruído de medição $\eta$ quanto a instabilidade da inversão em regiões de baixo Vega, e não constitui um posterior probabilístico sobre um único parâmetro $\sigma$. O painel central mostra o posterior B-PINN como distribuição coerente sobre um único $\sigma$ global: ao integrar todas as observações simultaneamente sob a restrição da EDP, a B-PINN produz uma estimativa com menor viés e um IC 95% com interpretação probabilística formal. O painel direito compara as métricas-resumo das duas abordagens: a dispersão do NR descreve a variabilidade das estimativas ponto a ponto, não a incerteza sobre $\sigma_\text{true}$; o IC B-PINN, ao contrário, é uma afirmação calibrada sobre onde $\sigma_\text{true}$ provavelmente se encontra, com garantia frequentista de cobertura validada empiricamente [5][10].

## 5.3 Mapa de Identificabilidade (grade η × N)
<a id="sec53"></a>

In [ ]:
# ── Mapa de identificabilidade η × N ─────────────────────────────────────────
print("Gerando mapa de identificabilidade (pode demorar alguns minutos)...")
eta_grid = [0.01, 0.03, 0.07]
N_grid   = [30, 60, 100]
sig_id   = 0.25
resultados_grade = []
for i, eta in enumerate(eta_grid):
    linha = []
    for j, N_d in enumerate(N_grid):
        r = treinar_bpinn_rapido(sig_id, eta=eta, N_d=N_d, n_epochs=5000)
        linha.append(r)
        ic = r["ic_95"]; cob = "✓" if r.get("cobertura_95",0) else "✗"
        print(f"  η={eta*100:.0f}%, N={N_d}: viés={r['vies']:.4f}, "
              f"largura={ic['alto']-ic['baixo']:.4f}, {cob}")
    resultados_grade.append(linha)

tabela = met.tabela_identificabilidade(eta_grid, N_grid, resultados_grade)
fig = plt_sde.fig_mapa_identificabilidade(tabela)
fig.show()

Gerando mapa de identificabilidade (pode demorar alguns minutos)...
  η=1%, N=30: viés=0.0480, largura=0.1247, ✓
  η=1%, N=60: viés=0.0456, largura=0.1230, ✓
  η=1%, N=100: viés=0.0449, largura=0.1225, ✓
  η=3%, N=30: viés=0.0448, largura=0.1314, ✓
  η=3%, N=60: viés=0.0485, largura=0.1352, ✓
  η=3%, N=100: viés=0.0450, largura=0.1246, ✓
  η=7%, N=30: viés=0.0459, largura=0.1344, ✓
  η=7%, N=60: viés=0.0453, largura=0.1259, ✓
  η=7%, N=100: viés=0.0479, largura=0.1298, ✓


O mapa de identificabilidade desdobra-se em três painéis correspondentes às três métricas de qualidade do posterior. O heatmap de viés mostra que o erro sistemático permanece baixo em praticamente toda a grade, confirmando que a restrição física da EDP estabiliza a estimativa mesmo com poucos dados ou ruído elevado — o regularizador global opera independentemente do regime de informação. O heatmap de largura do IC 95% revela a transição entre regimes: células no canto de alto $\eta$ e baixo $N$ apresentam intervalos amplos (regime de baixa informação), enquanto células de baixo $\eta$ e alto $N$ produzem intervalos estreitos (regime de alta identificabilidade). O heatmap de cobertura empírica é o critério de calibração: células próximas de 95% atestam que o posterior B-PINN é epistemicamente honesto; desvios abaixo de 95% indicam subcobertura — posterior excessivamente confiante — e acima de 95% indicam sobrecobertura. O mapa serve como guia prático de viabilidade: dados o nível de ruído esperado no mercado e o número de opções líquidas disponíveis, é possível avaliar a priori a confiabilidade da estimativa de $\sigma$ [5][6].

<small>[↑ Voltar ao topo](#toc)</small>

## 5.4 Integração com Dados Reais de Mercado
<a id="sec54"></a>

In [ ]:
# ── Dados reais (yfinance) ou sintéticos calibrados ─────────────────────────
print("Tentando obter dados reais (SPY via yfinance)...")
dados_reais = dat.obter_dados_mercado("SPY")
if dados_reais is None:
    print("yfinance indisponível. Usando smile sintético calibrado (skew típico de equities).")
    dados_reais = dat._smile_mercado_tipico(S0=490., tau=60./365., r=0.05)
    print(f"  Ticker: {dados_reais['ticker']} | N={len(dados_reais['K'])}")

dr     = dados_reais
IV_r   = np.array([bs.iv_newton(c,s,k,0.05,t)
                   for c,s,k,t in zip(dr["C_obs"],dr["S"],dr["K"],dr["tau"])])
valid  = ~np.isnan(IV_r) & (IV_r>0.03) & (IV_r<2.0)
IV_atm = float(np.nanmedian(IV_r[valid]))
print(f"\nIV mediana (ATM): {IV_atm:.2%}  |  IV min/max: {IV_r[valid].min():.2%}/{IV_r[valid].max():.2%}")

mn_r = np.log(dr["K"][valid]/dr["S0"])
fig  = make_subplots(rows=1, cols=2,
    subplot_titles=["Superfície de Volatilidade Implícita",
                    "Ajuste Black-Scholes (IV mediana)"])
sc = go.Scatter(x=mn_r.tolist(), y=(IV_r[valid]*100).tolist(), mode="markers",
    marker=dict(color=dr["tau"][valid].tolist(), colorscale="Viridis",
                size=9, colorbar=dict(title="τ (anos)", x=0.44)),
    name="IV Mercado")
fig.add_trace(sc, row=1, col=1)
fig.update_xaxes(title_text="Log-moneyness ln(K/S₀)", row=1, col=1)
fig.update_yaxes(title_text="IV (%)", row=1, col=1)
K_fit = np.linspace(dr["K"][valid].min(), dr["K"][valid].max(), 200)
C_fit = bs.bs_call(dr["S0"], K_fit, 0.05, IV_atm, dr["tau_val"])
fig.add_trace(go.Scatter(x=dr["K"][valid].tolist(), y=dr["C_obs"][valid].tolist(),
    mode="markers", marker=dict(color="#6c757d",size=7,opacity=0.7),
    name="Preços observados"), row=1, col=2)
fig.add_trace(go.Scatter(x=K_fit.tolist(), y=C_fit.tolist(), mode="lines",
    line=dict(color="#264653",width=2.8),
    name=f"BS (IV_ATM={IV_atm:.1%})"), row=1, col=2)
fig.update_xaxes(title_text="Strike K", row=1, col=2)
fig.update_yaxes(title_text="Preço da Opção", row=1, col=2)
fig.update_layout(height=450, template="plotly_white",
    title_text=f"<b>Figura 12</b> — Dados de Mercado: {dr['ticker']}",
    font=dict(family="Times New Roman, serif", size=13))
fig.show()

Tentando obter dados reais (SPY via yfinance)...
  [yfinance] erro: No module named 'yfinance'
yfinance indisponível. Usando smile sintético calibrado (skew típico de equities).
  Ticker: SPY (sintético calibrado) | N=20

IV mediana (ATM): 17.39%  |  IV min/max: 12.90%/25.63%


O painel esquerdo exibe o smile/skew de volatilidade empírico: a IV não é constante, mas cresce para opções com log-moneyness $\ln(K/S_0)$ negativo (puts OTM), refletindo a assimetria da demanda por proteção contra quedas — fenômeno documentado empiricamente desde o crash de 1987 e diretamente incompatível com o modelo BS de $\sigma$ constante. A codificação de cores por maturidade $\tau$ revela ainda a estrutura a termo da volatilidade: IVs de curto prazo tendem a ser mais elevadas, capturando o prêmio de risco de volatilidade de curto prazo [3][4]. O painel direito confronta os preços de mercado com o ajuste BS utilizando a IV mediana ATM: o acordo é razoável próximo ao dinheiro mas deteriora progressivamente nas pontas, quantificando a inadequação de um único escalar $\sigma$ para descrever toda a superfície — limitação que motiva diretamente as extensões de Dupire e Heston discutidas na Seção 6.1.

<small>[↑ Voltar ao topo](#toc)</small>

---
# 6. Extensões e Limitações
<a id="parte6"></a>

## 6.1 Limitações do Black-Scholes com σ Constante
<a id="sec61"></a>

O modelo de Black-Scholes com volatilidade constante é matematicamente elegante mas empiricamente limitado em três frentes. Em primeiro lugar, a estrutura de _smile_ da volatilidade implícita — documentada extensamente desde o crash de 1987 — demonstra que diferentes strikes e maturidades implicam diferentes valores de $\sigma$, o que é incompatível com o modelo. Em segundo lugar, retornos reais exibem caudas pesadas (excesso de curtose positivo) que o GBM com distribuição normal não captura. Em terceiro, saltos discretos em preços durante eventos de mercado (anúncios de resultados, crises) introduzem descontinuidades que o processo de difusão contínua ignora.

Extensões naturais que a abordagem B-PINN pode acomodar com modificações relativamente diretas:

1. Volatilidade Local de Dupire [[8]](#referencias) generaliza $\sigma$ para uma função $\sigma(S,t)$, determinada pela equação de Dupire $\sigma^2_{\text{loc}}(K,T) = [\partial_T C - rK\partial_K C + rC] / [\frac{1}{2}K^2\partial_{KK}C]$. A B-PINN pode ser adaptada para recuperar $\sigma(S,\tau)$ como uma função suave em vez de um escalar — um problema inverso funcional em lugar do problema escalar deste projeto.

2. Modelo de Heston [[9]](#referencias) introduz volatilidade estocástica:
$$
dS_t = rS_t\,dt + \sqrt{v_t}\,S_t\,dW_t^S, \qquad dv_t = \kappa(\bar{v}-v_t)\,dt + \xi\sqrt{v_t}\,dW_t^v,
$$
com $\text{Cor}(dW^S, dW^v) = \rho\,dt$. A B-PINN para Heston recuperaria o quinteto $(\kappa, \bar{v}, \xi, \rho, v_0)$ — um problema inverso de dimensão 5 com a EDP de Heston como restrição física.

In [ ]:
# ── Demonstração: smile não capturado por BS com σ constante ─────────────────
K_ext = np.linspace(80, 120, 30)
sig_smile, C_limpo_ext, C_obs_ext = bs.smile_sintetico(100., K_ext, tau=0.25, eta=0.04, seed=99)
IV_ext = np.array([bs.iv_newton(c,100.,k,0.05,0.25) for c,k in zip(C_obs_ext,K_ext)])
sig_flat = float(np.nanmedian(IV_ext))
C_flat   = bs.bs_call(np.full(len(K_ext),100.), K_ext, 0.05, sig_flat, 0.25)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["<b>Limitação do BS:</b> smile vs. σ constante",
                    "Ajuste de preços: BS flat vs. smile real"])
fig.add_trace(go.Scatter(x=K_ext.tolist(), y=(sig_smile*100).tolist(), mode="lines",
    line=dict(color="#264653",width=2.8), name="σ(K) verdadeiro"), row=1,col=1)
fig.add_trace(go.Scatter(x=K_ext.tolist(), y=(IV_ext*100).tolist(), mode="markers",
    marker=dict(color="#e76f51",size=7), name="IV pontual"), row=1,col=1)
fig.add_hline(y=sig_flat*100, line_dash="dash", line_color="#8d99ae", line_width=2,
    annotation_text=f"σ flat = {sig_flat:.1%}", row=1,col=1)
fig.add_trace(go.Scatter(x=K_ext.tolist(), y=C_obs_ext.tolist(), mode="markers",
    marker=dict(color="#8d99ae",size=6,opacity=0.7), name="Observado"), row=1,col=2)
fig.add_trace(go.Scatter(x=K_ext.tolist(), y=C_limpo_ext.tolist(), mode="lines",
    line=dict(color="#264653",width=2.8), name="Verdadeiro (smile)"), row=1,col=2)
fig.add_trace(go.Scatter(x=K_ext.tolist(), y=C_flat.tolist(), mode="lines",
    line=dict(color="#e76f51",width=2,dash="dash"), name="BS flat"), row=1,col=2)
fig.update_layout(height=450, template="plotly_white",
    title_text="<b>Figura 13</b> — Limitação do BS com σ constante",
    font=dict(family="Times New Roman, serif", size=13))
fig.show()
err_m = met.erro_relativo_precos(C_flat, C_obs_ext, C_limpo_ext)
print(f"BS flat: MAE vs. smile = {err_m['mae_limpo']:.4f} | MAPE = {err_m['mape_limpo']:.2f}%")

BS flat: MAE vs. smile = 0.2645 | MAPE = 18.88%


O painel esquerdo exibe a incompatibilidade estrutural: o smile sintético $\sigma(K)$ varia com o strike, enquanto o BS com $\sigma$ constante presume uma linha horizontal. O melhor $\sigma_\text{flat}$ possível (mediana das IVs pontuais) subestima a IV nas pontas e a superestima próximo ao ATM — um desvio sistemático que não pode ser eliminado por mais dados, pois é consequência da má-especificação do modelo e não do estimador. O painel direito traduz esse desvio em termos de preços: o BS com $\sigma_\text{flat}$ diverge das curvas do smile verdadeiro em opções OTM e ITM, com o erro MAPE refletindo diretamente essa má-especificação. Em tal cenário, a B-PINN estimará o $\sigma$ efetivo que minimiza o desajuste médio global — uma resposta internamente consistente com a física imposta —, mas sem interpretação econômica direta. A extensão natural é a volatilidade local de Dupire [8], que recupera a compatibilidade com o smile ao custo de um problema inverso funcional em vez do escalar tratado neste projeto.

<small>[↑ Voltar ao topo](#toc)</small>

## 6.2 Extensibilidade da Abordagem B-PINN
<a id="sec62"></a>

O _framework_ desenvolvido neste projeto é, em essência, um método geral de identificação de parâmetros de EDPs e SDEs a partir de dados esparsos com quantificação de incerteza. A tabela abaixo mostra sua universalidade:

| Domínio              | EDP/SDE          | Parâmetros          | Dados                |
|----------------------|------------------|---------------------|----------------------|
| Este projeto         | Black-Scholes    | σ (escalar)         | Preços de opções     |
| Finanças avançadas   | Black-Scholes    | σ(S,t) (função)     | Superfície de opções |
| Finanças avançadas   | Heston           | (κ, v̄, ξ, ρ, v₀)    | Opções + VIX         |
| Física computacional | Equação do calor | κ(x) (difusividade) | Dados de temperatura |
| Dinâmica de fluidos  | Navier-Stokes    | ν (viscosidade)     | Dados de velocidade  |
| Epidemiologia        | SIR estocástico  | (β, γ)              | Dados de contágio    |

Em todos os casos, o procedimento é o mesmo: 
1. Parametrização da solução por uma BNN; 
2. Adição dos parâmetros desconhecidos como variáveis latentes com _prior_; 
3. Enforque da EDP via pontos de colocação;
4. Ajuste aos dados via _likelihood_;
5. Inferência do _posterior_ via ELBO. 

O que muda é a EDP e o _prior_ dos parâmetros. [[5]](#referencias) [[10]](#referencias)

<small>[↑ Voltar ao topo](#toc)</small>

---
# Conclusão
<a id="conclusao"></a>


### Respondendo à Questão Central

> *"Até que ponto a volatilidade implícita é identificável a partir de dados ruidosos sob restrições físicas?"*

Os experimentos da Parte 5 permitem articular uma resposta quantitativa em quatro proposições:

**1. A identificabilidade existe, mas é condicionada ao regime de informação.** Para ruído baixo ($\eta \lesssim 3\%$) e quantidade moderada de observações ($N \gtrsim 50$), a B-PINN recupera $\sigma$ com viés pequeno e IC 95% calibrado. Fora desse regime, o _posterior_ se alarga progressivamente, refletindo a redução da informação disponível.

**2. A restrição física (EDP) é um regularizador global essencial.** Ao contrário da inversão ponto a ponto de Newton-Raphson, a EDP conecta preços em toda a superfície $(S, K, \tau)$: um ruído localizado num _strike_ específico é corrigido pela consistência global com a física do modelo. Isso explica o menor viés e maior estabilidade da B-PINN em comparação com o NR.

**3. A quantificação de incerteza é a contribuição única e diferenciadora da B-PINN.** Onde o Newton-Raphson entrega um número sem medida de confiança, a B-PINN entrega uma distribuição completa. Em regiões de baixo Vega — onde o dado é intrinsecamente não-informativo — o _posterior_ largo *é* a resposta correta: expressa honestamente a ignorância residual.

**4. O limite absoluto do método coincide com o limite do modelo.** Quando os dados exibem _smile_ ou _skew_ pronunciado, o pressuposto de $\sigma$ constante é incorretamente especificado. Nesse caso, a B-PINN estima um $\sigma$ "efetivo" que minimiza o desajuste médio, mas que não tem interpretação direta. A extensão para volatilidade local é o caminho natural.

### Tabela Comparativa Final

| Método | Resolve EDP | Estima σ | Incerteza | Dados Ruidosos | Interpretação |
|---|:---:|:---:|:---:|:---:|---|
| CN + Newton-Raphson | ✓ | Pontual | ✗ | Instável | Sem incerteza |
| PINN padrão (inverso) | ✓ | Pontual | ✗ | Parcial | Sem incerteza |
| **Bayesian PINN** | **✓** | **Posterior** | **✓** | **✓** | **Calibrada** |

### Mapa de Identificabilidade (Resumo)

| Condição | Identificabilidade | Posterior B-PINN |
|---|---|---|
| η ≈ 0%, N ≥ 50 | Alta | Concentrado em σ_true |
| η ≈ 3%, N ≥ 80 | Moderada | IC 95% cobre σ_true |
| η ≥ 7%, qualquer N | Baixa | Posterior largo, IC amplo |
| Deep OTM/ITM (Vega ≈ 0) | Baixa | Posterior difuso mesmo sem ruído |
| Modelo mal-especificado | N/A | Posterior em σ "efetivo" |

<small>[↑ Voltar ao topo](#toc)</small>

---
# Referências
<a id="referencias"></a>

<a id="BS1973"></a>**[1]** Black, F. & Scholes, M. (1973). *The Pricing of Options and Corporate Liabilities.* Journal of Political Economy, 81(3), 637–654. — Artigo original que introduziu a EDP de Black-Scholes e a fórmula analítica.

<a id="KS91"></a>**[2]** Karatzas, I. & Shreve, S. E. (1991). *Brownian Motion and Stochastic Calculus* (2ª ed.). Springer. — Referência rigorosa padrão para processos de Wiener, integrais de Itô e SDEs.

<a id="R2007"></a>**[3]** Rebonato, R. (2004). *Volatility and Correlation: The Perfect Hedger and the Fox.* John Wiley & Sons. — Discussão profunda sobre calibração de modelos de volatilidade a dados de mercado.

<a id="GK96"></a>**[4]** Gatheral, J. (2006). *The Volatility Surface: A Practitioner's Guide.* John Wiley & Sons. — Referência padrão sobre a estrutura empírica da superfície de volatilidade.

<a id="RPK2019"></a>**[5]** Raissi, M., Perdikaris, P. & Karniadakis, G. E. (2019). *Physics-Informed Neural Networks: A Deep Learning Framework for Solving Forward and Inverse Problems Involving Nonlinear Partial Differential Equations.* Journal of Computational Physics, 378, 686–707. — Artigo seminal das PINNs, com aplicações a problemas diretos e inversos.

<a id="H1902"></a>**[6]** Hadamard, J. (1902). *Sur les problèmes aux dérivées partielles et leur signification physique.* Princeton University Bulletin, 49–52. — Artigo original que introduziu os critérios de bem-posedness.

<a id="BBB2015"></a>**[7]** Blundell, C., Cornebise, J., Kavukcuoglu, K. & Wierstra, D. (2015). *Weight Uncertainty in Neural Networks.* Proceedings of ICML 2015. — Introdução do método Bayes by Backprop (reparametrização para VI em redes neurais).

<a id="D1994"></a>**[8]** Dupire, B. (1994). *Pricing with a Smile.* Risk Magazine, 7(1), 18–20. — Artigo original da volatilidade local, derivação da equação de Dupire.

<a id="H1993"></a>**[9]** Heston, S. L. (1993). *A Closed-Form Solution for Options with Stochastic Volatility with Applications to Bond and Currency Options.* Review of Financial Studies, 6(2), 327–343. — Modelo de Heston com solução semi-analítica via transformada de Fourier.

<a id="YWGK2020"></a>**[10]** Yang, L., Meng, X. & Karniadakis, G. E. (2021). *B-PINNs: Bayesian Physics-Informed Neural Networks for Forward and Inverse PDE Problems with Noisy Data.* Journal of Computational Physics, 425, 109913. — Artigo de referência para a B-PINN, diretamente relacionado ao projeto.

<a id="M1973"></a>**[11]** Merton, R. C. (1973). *Theory of Rational Option Pricing.* Bell Journal of Economics and Management Science, 4(1), 141–183. — Generalização e rigorização matemática do argumento de Black e Scholes.

<a id="O1998"></a>**[12]** Øksendal, B. (2003). *Stochastic Differential Equations: An Introduction with Applications* (6ª ed.). Springer. — Introdução acessível ao cálculo de Itô, indicada para graduação avançada.


> **Observação:** Esse Jupyter Notebook foi desenvolvido com o apoio de diferentes serviços de Inteligência Artificial a fim de melhor orientar a produção e manter a adequação técnico-formal do conteúdo relativo ao Cálculo Estocástico, Equações Diferenciais Estocásticas e o Modelo de Black-Scholes.

<small>[↑ Voltar ao topo](#toc)</small>